# 1. Plasma biomarker preprocessing

This notebook is part of the reproducible data-preparation pipeline used by the downstream modelling notebooks.


## 1.1. Connect Google Drive

first connect this notebook to Google Drive so that I can access the raw plasma biomarker file and save all cleaned datasets, quality-control outputs, and processing summaries within the existing ADNI non-imaging directory structure.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

## 1.2. Define the plasma data paths

define the path to the raw plasma biomarker file and create separate output folders for intermediate data, processed data, and quality-control results. then verify that the source file exists before continuing.

In [ ]:
from pathlib import Path

# Define the main non-imaging data directory.
base_dir = Path("/content/drive/MyDrive/adni_mri/adni_non_imaging")

# Define the raw plasma biomarker file.
plasma_raw_path = (
    base_dir
    / "raw"
    / "Plasma biomarker panel"
    / "All_Subjects_UPENN_PLASMA_FUJIREBIO_QUANTERIX_11Jul2026.csv"
)

# Define output directories for this modality.
plasma_interim_dir = base_dir / "interim" / "plasma"
plasma_processed_dir = base_dir / "processed" / "plasma"
plasma_qc_dir = base_dir / "qc" / "plasma"

# Create the output directories if they do not already exist.
plasma_interim_dir.mkdir(parents=True, exist_ok=True)
plasma_processed_dir.mkdir(parents=True, exist_ok=True)
plasma_qc_dir.mkdir(parents=True, exist_ok=True)

# Confirm that the raw source file is available.
if not plasma_raw_path.exists():
    raise FileNotFoundError(
        f"The plasma biomarker file was not found at:\n{plasma_raw_path}"
    )

print("Plasma source file found successfully.")
print(f"Raw file: {plasma_raw_path}")
print(f"Interim outputs: {plasma_interim_dir}")
print(f"Processed outputs: {plasma_processed_dir}")
print(f"QC outputs: {plasma_qc_dir}")

## 1.3. Load and inspect the raw plasma biomarker dataset

load the plasma biomarker CSV without modifying the original file. then inspect its dimensions, column names, data types, and a small sample of records to understand its structure before making any cleaning decisions.

In [ ]:
import pandas as pd

# Load the raw plasma biomarker dataset into memory.
plasma_raw = pd.read_csv(plasma_raw_path, low_memory=False)

# Display the overall dataset dimensions.
print(f"Number of rows: {plasma_raw.shape[0]:,}")
print(f"Number of columns: {plasma_raw.shape[1]:,}")

# Display every column name and its current data type.
plasma_structure = pd.DataFrame(
    {
        "COLUMN_NAME": plasma_raw.columns,
        "DATA_TYPE": plasma_raw.dtypes.astype(str).values,
    }
)

print("\nDataset structure:")
display(plasma_structure)

# Display the first five records without modifying the raw dataset.
print("\nFirst five records:")
display(plasma_raw.head())

## 1.4. Examine plasma biomarker availability and special values

summarise the availability of every plasma biomarker and derived ratio. For each column, count the total non-missing values, positive measurements, zero values, negative values, and distinct negative codes.

This is especially important because values such as `-4` cannot represent genuine biomarker concentrations and are likely to indicate that a measurement was unavailable or not performed. At this stage, only inspect these values and will not yet replace or remove them.

In [ ]:
# Define the direct biomarker measurements and supplied ratios.
plasma_marker_columns = [
    "pT217_F",
    "AB42_F",
    "AB40_F",
    "AB42_AB40_F",
    "pT217_AB42_F",
    "NfL_Q",
    "GFAP_Q",
    "NfL_F",
    "GFAP_F",
]

# Build a complete availability summary for every biomarker field.
marker_availability_rows = []

for column in plasma_marker_columns:
    values = plasma_raw[column]

    negative_values = sorted(
        values.loc[values < 0].dropna().unique().tolist()
    )

    marker_availability_rows.append(
        {
            "MARKER": column,
            "TOTAL_ROWS": len(values),
            "NON_MISSING": values.notna().sum(),
            "POSITIVE_VALUES": (values > 0).sum(),
            "ZERO_VALUES": (values == 0).sum(),
            "NEGATIVE_VALUES": (values < 0).sum(),
            "MISSING_NAN": values.isna().sum(),
            "UNIQUE_NEGATIVE_CODES": negative_values,
            "MINIMUM_RECORDED_VALUE": values.min(),
            "MAXIMUM_RECORDED_VALUE": values.max(),
        }
    )

plasma_marker_availability = pd.DataFrame(marker_availability_rows)

print("Plasma biomarker and ratio availability:")
display(plasma_marker_availability)

# Report the supplied Aβ42/Aβ40 ratio availability explicitly.
abeta_ratio_non_missing = plasma_raw["AB42_AB40_F"].notna().sum()
abeta_ratio_positive = (plasma_raw["AB42_AB40_F"] > 0).sum()

print(
    "\nSupplied Aβ42/Aβ40 ratio records:"
    f"\n  Non-missing values: {abeta_ratio_non_missing:,}"
    f"\n  Positive candidate measurements: {abeta_ratio_positive:,}"
)

## 1.5. Retrieve the plasma variable definitions from the ADNI data dictionary

search the ADNI data dictionary for all variables contained in the plasma biomarker dataset. This will allow me to verify the official marker descriptions, assay-related fields, units, valid ranges, and the meanings of special values such as `-4` and `-5` before modifying the data.

display all matching dictionary rows without compressing the output because a variable may have multiple entries describing its coding or phase-specific use.

In [ ]:
# Define the exact path to the ADNI data dictionary.
datadic_path = (
    base_dir
    / "raw"
    / "Cohort, dates and source-of-truth tables"
    / "DATADIC_11Jul2026.csv"
)

# Confirm that the dictionary file exists.
if not datadic_path.exists():
    raise FileNotFoundError(
        f"The ADNI data dictionary was not found at:\n{datadic_path}"
    )

print(f"Using data dictionary:\n{datadic_path}")

# Load the data dictionary without modifying the source file.
datadic = pd.read_csv(datadic_path, low_memory=False)

# Define all plasma dataset columns requiring dictionary review.
plasma_dictionary_variables = [
    "PHASE",
    "PTID",
    "RID",
    "VISCODE",
    "VISCODE2",
    "EXAMDATE",
    "Primary",
    "Additive",
    "pT217_F",
    "AB42_F",
    "AB40_F",
    "AB42_AB40_F",
    "pT217_AB42_F",
    "NfL_Q",
    "GFAP_Q",
    "NfL_F",
    "GFAP_F",
    "Comment",
    "update_stamp",
]

# Identify the dictionary column containing variable names.
variable_name_candidates = [
    "FLDNAME",
    "VARIABLE",
    "VARIABLE_NAME",
    "COLUMN_NAME",
    "FIELD_NAME",
]

variable_name_column = next(
    (
        column
        for column in variable_name_candidates
        if column in datadic.columns
    ),
    None,
)

if variable_name_column is None:
    raise KeyError(
        "The variable-name column could not be identified in the data dictionary.\n"
        f"Available columns:\n{datadic.columns.tolist()}"
    )

# Match the plasma variables without relying on capitalisation.
target_names_upper = {
    variable.upper() for variable in plasma_dictionary_variables
}

plasma_dictionary = datadic.loc[
    datadic[variable_name_column]
    .astype("string")
    .str.strip()
    .str.upper()
    .isin(target_names_upper)
].copy()

# Preserve the original plasma dataset column order in the output.
variable_order = {
    variable.upper(): position
    for position, variable in enumerate(plasma_dictionary_variables)
}

plasma_dictionary["_VARIABLE_ORDER"] = (
    plasma_dictionary[variable_name_column]
    .astype("string")
    .str.strip()
    .str.upper()
    .map(variable_order)
)

plasma_dictionary = (
    plasma_dictionary
    .sort_values("_VARIABLE_ORDER")
    .drop(columns="_VARIABLE_ORDER")
    .reset_index(drop=True)
)

print(
    f"\nDictionary rows matched: {len(plasma_dictionary):,}"
    f"\nPlasma variables represented: "
    f"{plasma_dictionary[variable_name_column].nunique():,}"
    f" of {len(plasma_dictionary_variables):,}"
)

print("\nAll matching plasma dictionary entries:")
display(plasma_dictionary)

# Report any plasma columns without a direct dictionary match.
matched_variables_upper = set(
    plasma_dictionary[variable_name_column]
    .astype("string")
    .str.strip()
    .str.upper()
)

unmatched_variables = [
    variable
    for variable in plasma_dictionary_variables
    if variable.upper() not in matched_variables_upper
]

print("\nPlasma variables without a direct DATADIC match:")
print(unmatched_variables)

In [ ]:
# Define only the variables that are specific to this plasma dataset.
plasma_specific_variables = [
    "pT217_F",
    "AB42_F",
    "AB40_F",
    "AB42_AB40_F",
    "pT217_AB42_F",
    "NfL_Q",
    "GFAP_Q",
    "NfL_F",
    "GFAP_F",
]

# Identify likely dictionary columns containing the source table or dataset name.
table_name_candidates = [
    "TBLNAME",
    "TABLE_NAME",
    "TABLENAME",
    "DATASET",
    "DATASET_NAME",
    "FILE",
    "FILE_NAME",
]

table_name_column = next(
    (
        column
        for column in table_name_candidates
        if column in datadic.columns
    ),
    None,
)

if table_name_column is None:
    raise KeyError(
        "The table-name column could not be identified in the data dictionary.\n"
        f"Available columns:\n{datadic.columns.tolist()}"
    )

# Retrieve dictionary rows for the plasma-specific variables only.
plasma_specific_dictionary = datadic.loc[
    datadic[variable_name_column]
    .astype("string")
    .str.strip()
    .str.upper()
    .isin({variable.upper() for variable in plasma_specific_variables})
].copy()

# Summarise which ADNI tables contain these variables.
plasma_table_summary = (
    plasma_specific_dictionary
    .groupby(table_name_column, dropna=False)
    .agg(
        MATCHED_ROWS=(variable_name_column, "size"),
        MATCHED_VARIABLES=(variable_name_column, "nunique"),
        VARIABLES=(
            variable_name_column,
            lambda values: sorted(values.astype(str).unique().tolist()),
        ),
    )
    .reset_index()
    .sort_values(
        ["MATCHED_VARIABLES", "MATCHED_ROWS"],
        ascending=[False, False],
    )
)

print("Candidate dictionary tables for the plasma dataset:")
display(plasma_table_summary)

## 1.6. Review the official dictionary entries for the plasma dataset

I identified `UPENN_PLASMA_FUJIREBIO_QUANTERIX` as the exact data-dictionary table corresponding to the plasma biomarker file.

retrieve every dictionary row from this table. This should provide a compact, dataset-specific description of the identifiers, assay fields, biomarkers, ratios, comments, units, and any documented special-value codes.

In [ ]:
# Define the exact ADNI dictionary table corresponding to the plasma file.
plasma_table_name = "UPENN_PLASMA_FUJIREBIO_QUANTERIX"

# Extract only dictionary rows belonging to this plasma dataset.
plasma_dictionary = datadic.loc[
    datadic[table_name_column]
    .astype("string")
    .str.strip()
    .str.upper()
    .eq(plasma_table_name.upper())
].copy()

# Preserve the same variable order used in the raw plasma dataset where possible.
plasma_column_order = {
    column.upper(): position
    for position, column in enumerate(plasma_raw.columns)
}

plasma_dictionary["_COLUMN_ORDER"] = (
    plasma_dictionary[variable_name_column]
    .astype("string")
    .str.strip()
    .str.upper()
    .map(plasma_column_order)
)

plasma_dictionary = (
    plasma_dictionary
    .sort_values(
        ["_COLUMN_ORDER", variable_name_column],
        na_position="last",
    )
    .drop(columns="_COLUMN_ORDER")
    .reset_index(drop=True)
)

print(f"Dictionary table: {plasma_table_name}")
print(f"Dictionary rows retrieved: {len(plasma_dictionary):,}")
print(
    "Unique variables represented: "
    f"{plasma_dictionary[variable_name_column].nunique():,}"
)

print("\nAll plasma-specific dictionary entries:")
display(plasma_dictionary)

# Compare the raw plasma columns with the variables documented for this table.
documented_variables_upper = set(
    plasma_dictionary[variable_name_column]
    .astype("string")
    .str.strip()
    .str.upper()
)

undocumented_raw_columns = [
    column
    for column in plasma_raw.columns
    if column.upper() not in documented_variables_upper
]

print("\nRaw plasma columns without a dictionary entry:")
print(undocumented_raw_columns)

## 1.7. Summary of the plasma data-dictionary review

The plasma file corresponds to the ADNI data-dictionary table `UPENN_PLASMA_FUJIREBIO_QUANTERIX`, which covers ADNI1, ADNIGO, ADNI2, ADNI3, and ADNI4. The dictionary contains 18 of the 19 raw dataset columns; only `update_stamp` is not documented because it is an administrative timestamp rather than an analysis variable.

The samples are blood specimens (`Primary = BLD`) collected using EDTA as the additive (`Additive = EDT`). The dataset contains the following plasma biomarkers and derived ratios:

| Variable | Description | Assay platform | Unit |
|---|---|---|---|
| `pT217_F` | Phosphorylated tau at threonine 217 | Fujirebio | pg/mL |
| `AB42_F` | Amyloid-beta 42 | Fujirebio | pg/mL |
| `AB40_F` | Amyloid-beta 40 | Fujirebio | pg/mL |
| `AB42_AB40_F` | Amyloid-beta 42/40 ratio | Fujirebio | Unitless |
| `pT217_AB42_F` | p-tau217/Aβ42 ratio | Fujirebio | Unitless |
| `NfL_Q` | Neurofilament light chain | Quanterix | pg/mL |
| `GFAP_Q` | Glial fibrillary acidic protein | Quanterix | pg/mL |
| `NfL_F` | Neurofilament light chain | Fujirebio | pg/mL |
| `GFAP_F` | Glial fibrillary acidic protein | Fujirebio | pg/mL |

The suffix `_F` identifies measurements produced using the Fujirebio platform, while `_Q` identifies measurements produced using the Quanterix platform. `NfL` and `GFAP` are therefore available from two different assay platforms and must remain as separate features rather than being treated as interchangeable measurements.

The dictionary confirms that negative values are administrative missing-value codes rather than valid biomarker concentrations:

- `-4` means **insufficient sample** for the Fujirebio p-tau217, Aβ42, Aβ40, and ratio fields.
- For Quanterix `NfL_Q` and `GFAP_Q`, `-4` means **insufficient sample**, while `-5` indicates that the sample quantity was not sufficient or available for the assay.
- For Fujirebio `NfL_F` and `GFAP_F`, `-4` is documented more generally as a **missing value**.

These negative codes must therefore be converted to missing values before statistical analysis or modelling. Their original reasons should still be preserved in quality-control summaries. The supplied `AB42_AB40_F` and `pT217_AB42_F` fields are unitless derived ratios and should later be checked against ratios recalculated from their component biomarker values.

## 1.8. Preserve and standardise the plasma biomarker values

create a separate working copy of the raw plasma dataset and convert the documented negative administrative codes to proper missing values.

Before replacing them, preserve boolean flags showing where `-4` or `-5` originally occurred. This allows the reason for unavailable measurements to remain traceable during quality-control reporting.

The raw source dataset will remain unchanged.

In [ ]:
import numpy as np

# Create a separate working copy so that the raw dataset remains unchanged.
plasma_cleaned = plasma_raw.copy()

# Standardise identifier, visit, sample-description, and comment fields.
text_columns = [
    "PHASE",
    "PTID",
    "VISCODE",
    "VISCODE2",
    "Primary",
    "Additive",
    "Comment",
]

for column in text_columns:
    plasma_cleaned[column] = (
        plasma_cleaned[column]
        .astype("string")
        .str.strip()
    )

# Convert key identifier and date fields to appropriate data types.
plasma_cleaned["RID"] = pd.to_numeric(
    plasma_cleaned["RID"],
    errors="coerce",
).astype("Int64")

plasma_cleaned["EXAMDATE"] = pd.to_datetime(
    plasma_cleaned["EXAMDATE"],
    errors="coerce",
)

plasma_cleaned["update_stamp"] = pd.to_datetime(
    plasma_cleaned["update_stamp"],
    errors="coerce",
)

# Ensure all biomarker and ratio fields are numeric.
for column in plasma_marker_columns:
    plasma_cleaned[column] = pd.to_numeric(
        plasma_cleaned[column],
        errors="coerce",
    )

# Preserve flags for every documented administrative missing-value code.
for column in plasma_marker_columns:
    plasma_cleaned[f"{column}_CODE_MINUS4"] = (
        plasma_cleaned[column] == -4
    )
    plasma_cleaned[f"{column}_CODE_MINUS5"] = (
        plasma_cleaned[column] == -5
    )

# Replace the documented negative codes with proper missing values.
plasma_cleaned[plasma_marker_columns] = (
    plasma_cleaned[plasma_marker_columns]
    .replace([-4, -5], np.nan)
)

# Summarise the conversion of special codes.
special_code_summary = pd.DataFrame(
    {
        "MARKER": plasma_marker_columns,
        "MINUS4_REPLACED": [
            int(plasma_cleaned[f"{column}_CODE_MINUS4"].sum())
            for column in plasma_marker_columns
        ],
        "MINUS5_REPLACED": [
            int(plasma_cleaned[f"{column}_CODE_MINUS5"].sum())
            for column in plasma_marker_columns
        ],
        "VALID_VALUES_AFTER_CLEANING": [
            int(plasma_cleaned[column].notna().sum())
            for column in plasma_marker_columns
        ],
        "MISSING_AFTER_CLEANING": [
            int(plasma_cleaned[column].isna().sum())
            for column in plasma_marker_columns
        ],
    }
)

print("Documented plasma missing-value codes converted successfully:")
display(special_code_summary)

print("\nCleaned plasma dataset dimensions:")
print(f"Rows: {plasma_cleaned.shape[0]:,}")
print(f"Columns: {plasma_cleaned.shape[1]:,}")

## 1.9. Validate and recover the plasma Aβ42/Aβ40 ratio

compare the supplied `AB42_AB40_F` values with ratios recalculated directly from `AB42_F / AB40_F`.

This check will determine:

- how many rows have both a supplied and recalculated ratio;
- whether the supplied values agree with the recalculated values;
- whether any missing supplied ratios can be recovered from valid component measurements;
- whether any rows contain a supplied ratio despite one or both component biomarkers being unavailable.

preserve the original supplied ratio, create a separate recalculated ratio, and flag any disagreements before deciding how to construct the final cleaned ratio.

In [ ]:
# Recalculate the Aβ42/Aβ40 ratio only where both components are available
# and the denominator is strictly positive.
valid_ratio_components = (
    plasma_cleaned["AB42_F"].notna()
    & plasma_cleaned["AB40_F"].notna()
    & (plasma_cleaned["AB40_F"] > 0)
)

plasma_cleaned["AB42_AB40_F_RECALCULATED"] = np.nan

plasma_cleaned.loc[
    valid_ratio_components,
    "AB42_AB40_F_RECALCULATED",
] = (
    plasma_cleaned.loc[valid_ratio_components, "AB42_F"]
    / plasma_cleaned.loc[valid_ratio_components, "AB40_F"]
)

# Identify the row-level availability combinations.
supplied_ratio_available = plasma_cleaned["AB42_AB40_F"].notna()
recalculated_ratio_available = (
    plasma_cleaned["AB42_AB40_F_RECALCULATED"].notna()
)

both_ratios_available = (
    supplied_ratio_available & recalculated_ratio_available
)

recoverable_missing_ratio = (
    ~supplied_ratio_available & recalculated_ratio_available
)

supplied_without_components = (
    supplied_ratio_available & ~recalculated_ratio_available
)

# Calculate absolute and relative differences where both ratios are available.
plasma_cleaned["AB42_AB40_F_ABS_DIFFERENCE"] = np.nan
plasma_cleaned["AB42_AB40_F_REL_DIFFERENCE"] = np.nan

plasma_cleaned.loc[
    both_ratios_available,
    "AB42_AB40_F_ABS_DIFFERENCE",
] = (
    plasma_cleaned.loc[both_ratios_available, "AB42_AB40_F"]
    - plasma_cleaned.loc[
        both_ratios_available,
        "AB42_AB40_F_RECALCULATED",
    ]
).abs()

plasma_cleaned.loc[
    both_ratios_available,
    "AB42_AB40_F_REL_DIFFERENCE",
] = (
    plasma_cleaned.loc[
        both_ratios_available,
        "AB42_AB40_F_ABS_DIFFERENCE",
    ]
    / plasma_cleaned.loc[
        both_ratios_available,
        "AB42_AB40_F_RECALCULATED",
    ].abs()
)

# Use a small tolerance to allow for rounding in the supplied ratio.
absolute_tolerance = 1e-6
relative_tolerance = 1e-4

plasma_cleaned["AB42_AB40_F_DISAGREEMENT"] = False

plasma_cleaned.loc[
    both_ratios_available,
    "AB42_AB40_F_DISAGREEMENT",
] = (
    plasma_cleaned.loc[
        both_ratios_available,
        "AB42_AB40_F_ABS_DIFFERENCE",
    ]
    > absolute_tolerance
) & (
    plasma_cleaned.loc[
        both_ratios_available,
        "AB42_AB40_F_REL_DIFFERENCE",
    ]
    > relative_tolerance
)

# Summarise overlap and recoverability.
ratio_overlap_summary = pd.DataFrame(
    {
        "CHECK": [
            "Supplied ratio available",
            "Recalculated ratio available",
            "Both supplied and recalculated available",
            "Missing supplied ratio recoverable from components",
            "Supplied ratio present but components unavailable",
            "Supplied and recalculated ratios disagree",
        ],
        "ROWS": [
            int(supplied_ratio_available.sum()),
            int(recalculated_ratio_available.sum()),
            int(both_ratios_available.sum()),
            int(recoverable_missing_ratio.sum()),
            int(supplied_without_components.sum()),
            int(
                plasma_cleaned[
                    "AB42_AB40_F_DISAGREEMENT"
                ].sum()
            ),
        ],
    }
)

print("Aβ42/Aβ40 ratio overlap and validation summary:")
display(ratio_overlap_summary)

print("\nDifference summary where both ratios are available:")
display(
    plasma_cleaned.loc[
        both_ratios_available,
        [
            "AB42_AB40_F_ABS_DIFFERENCE",
            "AB42_AB40_F_REL_DIFFERENCE",
        ],
    ].describe()
)

print("\nRows where the supplied ratio can be recovered:")
display(
    plasma_cleaned.loc[
        recoverable_missing_ratio,
        [
            "PHASE",
            "PTID",
            "RID",
            "VISCODE2",
            "EXAMDATE",
            "AB42_F",
            "AB40_F",
            "AB42_AB40_F",
            "AB42_AB40_F_RECALCULATED",
            "Comment",
        ],
    ]
)

print("\nRows with meaningful supplied-versus-recalculated disagreement:")
display(
    plasma_cleaned.loc[
        plasma_cleaned["AB42_AB40_F_DISAGREEMENT"],
        [
            "PHASE",
            "PTID",
            "RID",
            "VISCODE2",
            "EXAMDATE",
            "AB42_F",
            "AB40_F",
            "AB42_AB40_F",
            "AB42_AB40_F_RECALCULATED",
            "AB42_AB40_F_ABS_DIFFERENCE",
            "AB42_AB40_F_REL_DIFFERENCE",
            "Comment",
        ],
    ].sort_values(
        "AB42_AB40_F_REL_DIFFERENCE",
        ascending=False,
    )
)

## 1.10. Interpret the Aβ42/Aβ40 ratio comparison

The supplied and recalculated Aβ42/Aβ40 ratios have identical availability: all 2,292 valid supplied ratios have valid Aβ42 and Aβ40 component measurements, and no missing supplied ratio can be recovered from the component values.

Although 1,247 rows were initially flagged as disagreements, these differences are caused by numerical rounding rather than conflicting measurements. The supplied ratio is commonly stored to four decimal places, whereas the recalculated ratio retains greater precision. The largest absolute difference is approximately $$\(5 \times 10^{-5}\)$$, which is consistent with rounding to four decimal places.

Therefore, the supplied `AB42_AB40_F` values will be retained as the primary ratio feature. The recalculated ratio will be used only for quality-control validation, with agreement assessed after rounding both values to four decimal places.

In [ ]:
# Reassess agreement using the precision shown in the supplied ratio values.
plasma_cleaned["AB42_AB40_F_AGREES_AT_4DP"] = False

plasma_cleaned.loc[
    both_ratios_available,
    "AB42_AB40_F_AGREES_AT_4DP",
] = (
    plasma_cleaned.loc[
        both_ratios_available,
        "AB42_AB40_F",
    ].round(4)
    ==
    plasma_cleaned.loc[
        both_ratios_available,
        "AB42_AB40_F_RECALCULATED",
    ].round(4)
)

ratio_rounding_summary = pd.DataFrame(
    {
        "CHECK": [
            "Both ratios available",
            "Agree after rounding to 4 decimal places",
            "Still disagree after rounding to 4 decimal places",
        ],
        "ROWS": [
            int(both_ratios_available.sum()),
            int(
                plasma_cleaned.loc[
                    both_ratios_available,
                    "AB42_AB40_F_AGREES_AT_4DP",
                ].sum()
            ),
            int(
                (
                    both_ratios_available
                    & ~plasma_cleaned["AB42_AB40_F_AGREES_AT_4DP"]
                ).sum()
            ),
        ],
    }
)

print("Aβ42/Aβ40 ratio agreement after accounting for rounding:")
display(ratio_rounding_summary)

print("\nRows still disagreeing after rounding to four decimal places:")
display(
    plasma_cleaned.loc[
        both_ratios_available
        & ~plasma_cleaned["AB42_AB40_F_AGREES_AT_4DP"],
        [
            "PHASE",
            "PTID",
            "RID",
            "VISCODE2",
            "EXAMDATE",
            "AB42_F",
            "AB40_F",
            "AB42_AB40_F",
            "AB42_AB40_F_RECALCULATED",
            "AB42_AB40_F_ABS_DIFFERENCE",
            "Comment",
        ],
    ]
)

## 1.11. Validate the plasma p-tau217/Aβ42 ratio

compare the supplied `pT217_AB42_F` values with ratios recalculated from `pT217_F / AB42_F`.

examine whether the supplied ratio has the same coverage as its component biomarkers, whether any missing ratios can be recovered, and whether the supplied values agree with the recalculated values after accounting for numerical rounding.

In [ ]:
# Recalculate the p-tau217/Aβ42 ratio only where both component
# measurements are available and Aβ42 is strictly positive.
valid_ptau_ratio_components = (
    plasma_cleaned["pT217_F"].notna()
    & plasma_cleaned["AB42_F"].notna()
    & (plasma_cleaned["AB42_F"] > 0)
)

plasma_cleaned["pT217_AB42_F_RECALCULATED"] = np.nan

plasma_cleaned.loc[
    valid_ptau_ratio_components,
    "pT217_AB42_F_RECALCULATED",
] = (
    plasma_cleaned.loc[valid_ptau_ratio_components, "pT217_F"]
    / plasma_cleaned.loc[valid_ptau_ratio_components, "AB42_F"]
)

# Identify availability overlap.
supplied_ptau_ratio_available = plasma_cleaned["pT217_AB42_F"].notna()

recalculated_ptau_ratio_available = (
    plasma_cleaned["pT217_AB42_F_RECALCULATED"].notna()
)

both_ptau_ratios_available = (
    supplied_ptau_ratio_available
    & recalculated_ptau_ratio_available
)

recoverable_missing_ptau_ratio = (
    ~supplied_ptau_ratio_available
    & recalculated_ptau_ratio_available
)

supplied_ptau_without_components = (
    supplied_ptau_ratio_available
    & ~recalculated_ptau_ratio_available
)

# Calculate numerical differences where both ratios are available.
plasma_cleaned["pT217_AB42_F_ABS_DIFFERENCE"] = np.nan

plasma_cleaned.loc[
    both_ptau_ratios_available,
    "pT217_AB42_F_ABS_DIFFERENCE",
] = (
    plasma_cleaned.loc[
        both_ptau_ratios_available,
        "pT217_AB42_F",
    ]
    - plasma_cleaned.loc[
        both_ptau_ratios_available,
        "pT217_AB42_F_RECALCULATED",
    ]
).abs()

# The supplied ratio is stored with greater precision than the Aβ42/Aβ40
# ratio, so agreement is checked at six decimal places.
plasma_cleaned["pT217_AB42_F_AGREES_AT_6DP"] = False

plasma_cleaned.loc[
    both_ptau_ratios_available,
    "pT217_AB42_F_AGREES_AT_6DP",
] = (
    plasma_cleaned.loc[
        both_ptau_ratios_available,
        "pT217_AB42_F",
    ].round(6)
    ==
    plasma_cleaned.loc[
        both_ptau_ratios_available,
        "pT217_AB42_F_RECALCULATED",
    ].round(6)
)

ptau_ratio_validation_summary = pd.DataFrame(
    {
        "CHECK": [
            "Supplied ratio available",
            "Recalculated ratio available",
            "Both supplied and recalculated available",
            "Missing supplied ratio recoverable from components",
            "Supplied ratio present but components unavailable",
            "Agree after rounding to 6 decimal places",
            "Still disagree after rounding to 6 decimal places",
        ],
        "ROWS": [
            int(supplied_ptau_ratio_available.sum()),
            int(recalculated_ptau_ratio_available.sum()),
            int(both_ptau_ratios_available.sum()),
            int(recoverable_missing_ptau_ratio.sum()),
            int(supplied_ptau_without_components.sum()),
            int(
                plasma_cleaned.loc[
                    both_ptau_ratios_available,
                    "pT217_AB42_F_AGREES_AT_6DP",
                ].sum()
            ),
            int(
                (
                    both_ptau_ratios_available
                    & ~plasma_cleaned["pT217_AB42_F_AGREES_AT_6DP"]
                ).sum()
            ),
        ],
    }
)

print("p-tau217/Aβ42 ratio validation summary:")
display(ptau_ratio_validation_summary)

print("\nMaximum absolute supplied-versus-recalculated difference:")
print(
    plasma_cleaned.loc[
        both_ptau_ratios_available,
        "pT217_AB42_F_ABS_DIFFERENCE",
    ].max()
)

print("\nRows still disagreeing after rounding to six decimal places:")
display(
    plasma_cleaned.loc[
        both_ptau_ratios_available
        & ~plasma_cleaned["pT217_AB42_F_AGREES_AT_6DP"],
        [
            "PHASE",
            "PTID",
            "RID",
            "VISCODE2",
            "EXAMDATE",
            "pT217_F",
            "AB42_F",
            "pT217_AB42_F",
            "pT217_AB42_F_RECALCULATED",
            "pT217_AB42_F_ABS_DIFFERENCE",
            "Comment",
        ],
    ]
)

## 1.12. Interpret the p-tau217/Aβ42 ratio comparison

The supplied and recalculated p-tau217/Aβ42 ratios have identical availability. All 2,292 valid supplied ratios have valid p-tau217 and Aβ42 component measurements, and no missing ratio can be recovered from the component values.

The initial six-decimal-place comparison flagged 1,170 rows, but the maximum absolute difference was approximately \(5 \times 10^{-6}\). This is consistent with the supplied ratio being rounded to five decimal places rather than six.

Therefore, the supplied `pT217_AB42_F` field should be retained as the primary ratio feature. The recalculated ratio is required only for quality-control validation, with agreement assessed after rounding both values to five decimal places.

In [ ]:
# Reassess agreement using the precision visible in the supplied ratio.
plasma_cleaned["pT217_AB42_F_AGREES_AT_5DP"] = False

plasma_cleaned.loc[
    both_ptau_ratios_available,
    "pT217_AB42_F_AGREES_AT_5DP",
] = (
    plasma_cleaned.loc[
        both_ptau_ratios_available,
        "pT217_AB42_F",
    ].round(5)
    ==
    plasma_cleaned.loc[
        both_ptau_ratios_available,
        "pT217_AB42_F_RECALCULATED",
    ].round(5)
)

ptau_ratio_rounding_summary = pd.DataFrame(
    {
        "CHECK": [
            "Both ratios available",
            "Agree after rounding to 5 decimal places",
            "Still disagree after rounding to 5 decimal places",
        ],
        "ROWS": [
            int(both_ptau_ratios_available.sum()),
            int(
                plasma_cleaned.loc[
                    both_ptau_ratios_available,
                    "pT217_AB42_F_AGREES_AT_5DP",
                ].sum()
            ),
            int(
                (
                    both_ptau_ratios_available
                    & ~plasma_cleaned["pT217_AB42_F_AGREES_AT_5DP"]
                ).sum()
            ),
        ],
    }
)

print("p-tau217/Aβ42 ratio agreement after accounting for rounding:")
display(ptau_ratio_rounding_summary)

print("\nRows still disagreeing after rounding to five decimal places:")
display(
    plasma_cleaned.loc[
        both_ptau_ratios_available
        & ~plasma_cleaned["pT217_AB42_F_AGREES_AT_5DP"],
        [
            "PHASE",
            "PTID",
            "RID",
            "VISCODE2",
            "EXAMDATE",
            "pT217_F",
            "AB42_F",
            "pT217_AB42_F",
            "pT217_AB42_F_RECALCULATED",
            "pT217_AB42_F_ABS_DIFFERENCE",
            "Comment",
        ],
    ]
)

Everything is okay, all passed validation

## 1.13. Check identifiers, visit information, and duplicate plasma records

verify that the participant identifiers, visit codes, and examination dates are complete and internally consistent.

also check for duplicate records using progressively stricter keys:

- exact duplicate rows;
- repeated participant-visit combinations;
- repeated participant-visit-date combinations.

Repeated measurements are not automatically errors because the same participant may have longitudinal plasma samples. However, multiple rows for the same participant at the same visit and examination date may indicate duplicated records or repeat assays that require closer inspection.

In [ ]:
# Check missingness in the core identifying and visit fields.
key_columns = [
    "PHASE",
    "PTID",
    "RID",
    "VISCODE",
    "VISCODE2",
    "EXAMDATE",
    "Primary",
    "Additive",
]

key_field_summary = pd.DataFrame(
    {
        "COLUMN": key_columns,
        "MISSING_VALUES": [
            int(plasma_cleaned[column].isna().sum())
            for column in key_columns
        ],
        "UNIQUE_VALUES": [
            int(plasma_cleaned[column].nunique(dropna=True))
            for column in key_columns
        ],
    }
)

print("Identifier and visit-field completeness:")
display(key_field_summary)

# Check whether PTID and RID map consistently to one another.
ptid_to_rid_counts = (
    plasma_cleaned.groupby("PTID", dropna=False)["RID"]
    .nunique(dropna=True)
)

rid_to_ptid_counts = (
    plasma_cleaned.groupby("RID", dropna=False)["PTID"]
    .nunique(dropna=True)
)

inconsistent_ptids = ptid_to_rid_counts[ptid_to_rid_counts > 1]
inconsistent_rids = rid_to_ptid_counts[rid_to_ptid_counts > 1]

print("\nPTIDs linked to more than one RID:")
display(inconsistent_ptids)

print("\nRIDs linked to more than one PTID:")
display(inconsistent_rids)

# Identify exact duplicated rows from the original 19 source columns.
source_columns = plasma_raw.columns.tolist()

exact_duplicate_mask = plasma_cleaned.duplicated(
    subset=source_columns,
    keep=False,
)

# Identify repeated records using participant and visit keys.
participant_visit_columns = [
    "RID",
    "VISCODE2",
]

participant_visit_date_columns = [
    "RID",
    "VISCODE2",
    "EXAMDATE",
]

duplicate_participant_visit_mask = plasma_cleaned.duplicated(
    subset=participant_visit_columns,
    keep=False,
)

duplicate_participant_visit_date_mask = plasma_cleaned.duplicated(
    subset=participant_visit_date_columns,
    keep=False,
)

duplicate_summary = pd.DataFrame(
    {
        "DUPLICATE_CHECK": [
            "Exact duplicated source rows",
            "Rows sharing RID and VISCODE2",
            "Rows sharing RID, VISCODE2, and EXAMDATE",
        ],
        "ROWS_FLAGGED": [
            int(exact_duplicate_mask.sum()),
            int(duplicate_participant_visit_mask.sum()),
            int(duplicate_participant_visit_date_mask.sum()),
        ],
        "DUPLICATE_GROUPS": [
            int(
                plasma_cleaned.loc[exact_duplicate_mask, source_columns]
                .drop_duplicates()
                .shape[0]
            ),
            int(
                plasma_cleaned.loc[
                    duplicate_participant_visit_mask,
                    participant_visit_columns,
                ]
                .drop_duplicates()
                .shape[0]
            ),
            int(
                plasma_cleaned.loc[
                    duplicate_participant_visit_date_mask,
                    participant_visit_date_columns,
                ]
                .drop_duplicates()
                .shape[0]
            ),
        ],
    }
)

print("\nDuplicate-record summary:")
display(duplicate_summary)

print("\nRecords sharing RID, VISCODE2, and EXAMDATE:")
display(
    plasma_cleaned.loc[
        duplicate_participant_visit_date_mask,
        [
            "PHASE",
            "PTID",
            "RID",
            "VISCODE",
            "VISCODE2",
            "EXAMDATE",
            "Primary",
            "Additive",
            "pT217_F",
            "AB42_F",
            "AB40_F",
            "AB42_AB40_F",
            "pT217_AB42_F",
            "NfL_Q",
            "GFAP_Q",
            "NfL_F",
            "GFAP_F",
            "Comment",
        ],
    ].sort_values(
        ["RID", "VISCODE2", "EXAMDATE"]
    )
)

## 1.14. Inspect repeated participant-visit combinations

Three participant-visit combinations occur more than once, but none share the same examination date. inspect these six records together and calculate the number of days between records within each repeated `RID` and `VISCODE2` group.

This will help determine whether they represent genuine repeat blood collections, visit-code inconsistencies, or records that require resolution using the authoritative visit tables. No records will be removed at this stage.

In [ ]:
# Extract all records that share the same participant and translated visit code.
repeated_participant_visits = (
    plasma_cleaned.loc[
        duplicate_participant_visit_mask,
        [
            "PHASE",
            "PTID",
            "RID",
            "VISCODE",
            "VISCODE2",
            "EXAMDATE",
            "Primary",
            "Additive",
            "pT217_F",
            "AB42_F",
            "AB40_F",
            "AB42_AB40_F",
            "pT217_AB42_F",
            "NfL_Q",
            "GFAP_Q",
            "NfL_F",
            "GFAP_F",
            "Comment",
        ],
    ]
    .sort_values(["RID", "VISCODE2", "EXAMDATE"])
    .copy()
)

# Calculate the interval between consecutive records within each repeated group.
repeated_participant_visits["DAYS_FROM_PREVIOUS_RECORD"] = (
    repeated_participant_visits
    .groupby(["RID", "VISCODE2"])["EXAMDATE"]
    .diff()
    .dt.days
)

print("Repeated RID and VISCODE2 combinations:")
display(repeated_participant_visits)

# Summarise each repeated participant-visit group.
repeated_visit_group_summary = (
    repeated_participant_visits
    .groupby(["PTID", "RID", "VISCODE2"], dropna=False)
    .agg(
        RECORD_COUNT=("EXAMDATE", "size"),
        PHASES=("PHASE", lambda values: sorted(values.unique().tolist())),
        VISCODE_VALUES=(
            "VISCODE",
            lambda values: sorted(values.unique().tolist()),
        ),
        FIRST_EXAMDATE=("EXAMDATE", "min"),
        LAST_EXAMDATE=("EXAMDATE", "max"),
    )
    .reset_index()
)

repeated_visit_group_summary["DATE_SEPARATION_DAYS"] = (
    repeated_visit_group_summary["LAST_EXAMDATE"]
    - repeated_visit_group_summary["FIRST_EXAMDATE"]
).dt.days

print("\nSummary of repeated participant–visit groups:")
display(repeated_visit_group_summary)

Dates are different, no duplicates, all valid

## 1.15. Examine plasma biomarker coverage across ADNI phases

summarise the number of records and participants available in each ADNI phase and examine how biomarker availability differs by phase.

This is important because the Fujirebio and Quanterix measurements were not necessarily available for the same samples or study periods. The phase-level summary will show whether missingness reflects assay coverage rather than random data loss.

In [ ]:
# Summarise records and participant coverage by ADNI phase.
phase_record_summary = (
    plasma_cleaned
    .groupby("PHASE", dropna=False)
    .agg(
        RECORDS=("RID", "size"),
        PARTICIPANTS=("RID", "nunique"),
        FIRST_EXAMDATE=("EXAMDATE", "min"),
        LAST_EXAMDATE=("EXAMDATE", "max"),
    )
    .reset_index()
    .sort_values("PHASE")
)

print("Plasma dataset coverage by ADNI phase:")
display(phase_record_summary)

# Summarise valid biomarker availability within each phase.
phase_marker_availability = (
    plasma_cleaned
    .groupby("PHASE", dropna=False)[plasma_marker_columns]
    .agg(lambda values: values.notna().sum())
    .reset_index()
)

print("\nValid biomarker measurements by ADNI phase:")
display(phase_marker_availability)

# Calculate phase-specific percentages of available measurements.
phase_sizes = (
    plasma_cleaned
    .groupby("PHASE", dropna=False)
    .size()
)

phase_marker_percentages = (
    plasma_cleaned
    .groupby("PHASE", dropna=False)[plasma_marker_columns]
    .agg(lambda values: values.notna().mean() * 100)
    .round(2)
    .reset_index()
)

print("\nPercentage of records with each biomarker available:")
display(phase_marker_percentages)

# Summarise common assay-availability patterns across all records.
plasma_cleaned["HAS_FUJIREBIO_AMYLOID_PTAU"] = (
    plasma_cleaned[
        [
            "pT217_F",
            "AB42_F",
            "AB40_F",
            "AB42_AB40_F",
            "pT217_AB42_F",
        ]
    ]
    .notna()
    .all(axis=1)
)

plasma_cleaned["HAS_QUANTERIX_NFL_GFAP"] = (
    plasma_cleaned[["NfL_Q", "GFAP_Q"]]
    .notna()
    .all(axis=1)
)

plasma_cleaned["HAS_FUJIREBIO_NFL_GFAP"] = (
    plasma_cleaned[["NfL_F", "GFAP_F"]]
    .notna()
    .all(axis=1)
)

assay_pattern_summary = (
    plasma_cleaned
    .groupby(
        [
            "HAS_FUJIREBIO_AMYLOID_PTAU",
            "HAS_QUANTERIX_NFL_GFAP",
            "HAS_FUJIREBIO_NFL_GFAP",
        ],
        dropna=False,
    )
    .size()
    .reset_index(name="RECORDS")
    .sort_values("RECORDS", ascending=False)
)

print("\nCombined assay-availability patterns:")
display(assay_pattern_summary)

## 1.16. Inspect plasma comments and assay-related quality-control notes

examine all non-empty values in the `Comment` field and summarise how frequently each comment occurs.

This is necessary because the comments may identify batch effects, assay drift, repeated measurements, validation decisions, or other laboratory quality-control information that could affect whether particular biomarker results should be retained, flagged, or excluded.

In [ ]:
# Extract records containing a non-empty laboratory comment.
comment_mask = (
    plasma_cleaned["Comment"].notna()
    & plasma_cleaned["Comment"].str.strip().ne("")
)

commented_records = plasma_cleaned.loc[comment_mask].copy()

print(f"Records with a non-empty comment: {len(commented_records):,}")

# Summarise the exact comment values and their frequencies.
comment_summary = (
    commented_records
    .groupby("Comment", dropna=False)
    .agg(
        RECORDS=("RID", "size"),
        PARTICIPANTS=("RID", "nunique"),
        PHASES=("PHASE", lambda values: sorted(values.unique().tolist())),
        FIRST_EXAMDATE=("EXAMDATE", "min"),
        LAST_EXAMDATE=("EXAMDATE", "max"),
    )
    .reset_index()
    .sort_values("RECORDS", ascending=False)
)

# Temporarily allow pandas to display the complete comment text.
previous_max_colwidth = pd.get_option("display.max_colwidth")
pd.set_option("display.max_colwidth", None)

print("Complete unique plasma comments:")

for index, row in comment_summary.iterrows():
    print("\n" + "=" * 100)
    print(f"Comment {index + 1}")
    print("=" * 100)
    print(row["Comment"])
    print(f"\nRecords: {row['RECORDS']:,}")
    print(f"Participants: {row['PARTICIPANTS']:,}")
    print(f"Phases: {row['PHASES']}")
    print(f"Date range: {row['FIRST_EXAMDATE'].date()} to {row['LAST_EXAMDATE'].date()}")

# Restore the previous pandas display setting.
pd.set_option("display.max_colwidth", previous_max_colwidth)

# Display all commented records with their biomarker measurements.
print("\nAll records containing laboratory comments:")
display(
    commented_records[
        [
            "PHASE",
            "PTID",
            "RID",
            "VISCODE",
            "VISCODE2",
            "EXAMDATE",
            "pT217_F",
            "AB42_F",
            "AB40_F",
            "AB42_AB40_F",
            "pT217_AB42_F",
            "NfL_Q",
            "GFAP_Q",
            "NfL_F",
            "GFAP_F",
            "Comment",
        ]
    ].sort_values(
        ["Comment", "RID", "EXAMDATE"]
    )
)

## 1.17. Preserve the validated Batch 3 quality-control flag

The laboratory comment identifies 393 ADNI4 records from Batch 3 where assay drift was observed but the final results were validated.

Because the laboratory explicitly confirmed that these measurements are valid, retain them. create a separate boolean QC flag so that the affected records remain identifiable during later distribution checks, modelling, and sensitivity analysis.

In [ ]:
# Create a boolean flag for records carrying the validated Batch 3 QC note.
plasma_cleaned["BATCH3_QC_DRIFT_VALIDATED"] = (
    plasma_cleaned["Comment"]
    .fillna("")
    .str.contains(
        "Batch 3: QC drift noted; results validated",
        case=False,
        regex=False,
    )
)

# Summarise the flagged records overall and by phase.
batch3_qc_summary = pd.DataFrame(
    {
        "QC_STATUS": [
            "Batch 3 drift noted and results validated",
            "No Batch 3 validated-drift comment",
        ],
        "RECORDS": [
            int(plasma_cleaned["BATCH3_QC_DRIFT_VALIDATED"].sum()),
            int((~plasma_cleaned["BATCH3_QC_DRIFT_VALIDATED"]).sum()),
        ],
        "PARTICIPANTS": [
            int(
                plasma_cleaned.loc[
                    plasma_cleaned["BATCH3_QC_DRIFT_VALIDATED"],
                    "RID",
                ].nunique()
            ),
            int(
                plasma_cleaned.loc[
                    ~plasma_cleaned["BATCH3_QC_DRIFT_VALIDATED"],
                    "RID",
                ].nunique()
            ),
        ],
    }
)

print("Batch 3 validated-drift QC summary:")
display(batch3_qc_summary)

print("\nFlagged records by ADNI phase:")
display(
    plasma_cleaned.groupby("PHASE")[
        "BATCH3_QC_DRIFT_VALIDATED"
    ]
    .agg(
        FLAGGED_RECORDS="sum",
        TOTAL_RECORDS="size",
    )
    .reset_index()
)

print(
    "\nDecision: all records are retained because the laboratory "
    "comment confirms that the results were validated."
)

## 1.18. Inspect plasma biomarker distributions and possible extreme values

examine the cleaned distributions of each plasma biomarker and ratio using valid positive measurements only.

Because biomarker concentrations are usually right-skewed, calculate descriptive statistics on both the original scale and the natural-log scale. also flag possible extreme values using the \(1.5 \times IQR\) rule on the log-transformed values.

These flags are intended only for quality-control review. Values will not be removed automatically because clinically meaningful biomarker measurements can legitimately be very high or very low.

In [ ]:
# Define the biomarker and ratio fields to inspect.
distribution_columns = [
    "pT217_F",
    "AB42_F",
    "AB40_F",
    "AB42_AB40_F",
    "pT217_AB42_F",
    "NfL_Q",
    "GFAP_Q",
    "NfL_F",
    "GFAP_F",
]

distribution_summary_rows = []

for column in distribution_columns:
    valid_values = plasma_cleaned.loc[
        plasma_cleaned[column].notna()
        & (plasma_cleaned[column] > 0),
        column,
    ]

    log_values = np.log(valid_values)

    q1_log = log_values.quantile(0.25)
    q3_log = log_values.quantile(0.75)
    iqr_log = q3_log - q1_log

    lower_log_bound = q1_log - 1.5 * iqr_log
    upper_log_bound = q3_log + 1.5 * iqr_log

    lower_original_bound = np.exp(lower_log_bound)
    upper_original_bound = np.exp(upper_log_bound)

    outlier_mask = (
        plasma_cleaned[column].notna()
        & (
            (plasma_cleaned[column] < lower_original_bound)
            | (plasma_cleaned[column] > upper_original_bound)
        )
    )

    plasma_cleaned[f"{column}_LOG_IQR_OUTLIER"] = outlier_mask

    distribution_summary_rows.append(
        {
            "MARKER": column,
            "VALID_VALUES": int(valid_values.shape[0]),
            "MIN": valid_values.min(),
            "P01": valid_values.quantile(0.01),
            "Q1": valid_values.quantile(0.25),
            "MEDIAN": valid_values.median(),
            "Q3": valid_values.quantile(0.75),
            "P99": valid_values.quantile(0.99),
            "MAX": valid_values.max(),
            "LOG_IQR_LOWER_BOUND": lower_original_bound,
            "LOG_IQR_UPPER_BOUND": upper_original_bound,
            "POSSIBLE_EXTREME_VALUES": int(outlier_mask.sum()),
        }
    )

plasma_distribution_summary = pd.DataFrame(
    distribution_summary_rows
)

print("Plasma biomarker distribution summary:")
display(plasma_distribution_summary)

print("\nPossible extreme values by marker:")
display(
    plasma_distribution_summary[
        [
            "MARKER",
            "VALID_VALUES",
            "LOG_IQR_LOWER_BOUND",
            "LOG_IQR_UPPER_BOUND",
            "POSSIBLE_EXTREME_VALUES",
        ]
    ]
)

## 1.19. Inspect records flagged as possible plasma biomarker extremes

The log-scale IQR rule identified possible extreme observations across all plasma biomarkers. These flags do not prove that a value is erroneous, because genuine disease-related biomarker concentrations may occur in the tails of the distribution.

inspect the flagged records directly, including their participant information, examination date, assay platform, laboratory comment, and all related biomarker measurements. also explicitly identify values equal to the unusually round maxima of 1,000 pg/mL for Aβ42 and 5,000 pg/mL for Aβ40, as these may represent assay limits or special laboratory values requiring further review.

In [ ]:
# Combine all marker-specific extreme-value flags into one row-level flag.
outlier_flag_columns = [
    f"{column}_LOG_IQR_OUTLIER"
    for column in distribution_columns
]

plasma_cleaned["ANY_PLASMA_LOG_IQR_OUTLIER"] = (
    plasma_cleaned[outlier_flag_columns].any(axis=1)
)

# Record which markers caused each row to be flagged.
plasma_cleaned["PLASMA_OUTLIER_MARKERS"] = (
    plasma_cleaned[outlier_flag_columns]
    .apply(
        lambda row: [
            column.replace("_LOG_IQR_OUTLIER", "")
            for column, flagged in row.items()
            if flagged
        ],
        axis=1,
    )
)

# Explicitly flag the unusually round maximum amyloid values.
plasma_cleaned["AB42_F_EQUALS_1000"] = (
    plasma_cleaned["AB42_F"] == 1000
)

plasma_cleaned["AB40_F_EQUALS_5000"] = (
    plasma_cleaned["AB40_F"] == 5000
)

# Extract all records flagged by at least one distribution check.
plasma_extreme_records = plasma_cleaned.loc[
    plasma_cleaned["ANY_PLASMA_LOG_IQR_OUTLIER"],
    [
        "PHASE",
        "PTID",
        "RID",
        "VISCODE",
        "VISCODE2",
        "EXAMDATE",
        "pT217_F",
        "AB42_F",
        "AB40_F",
        "AB42_AB40_F",
        "pT217_AB42_F",
        "NfL_Q",
        "GFAP_Q",
        "NfL_F",
        "GFAP_F",
        "PLASMA_OUTLIER_MARKERS",
        "AB42_F_EQUALS_1000",
        "AB40_F_EQUALS_5000",
        "BATCH3_QC_DRIFT_VALIDATED",
        "Comment",
    ],
].copy()

print(
    "Records flagged by at least one log-IQR distribution check: "
    f"{len(plasma_extreme_records):,}"
)

print("\nNumber of flagged records by ADNI phase:")
display(
    plasma_extreme_records
    .groupby("PHASE", dropna=False)
    .size()
    .reset_index(name="FLAGGED_RECORDS")
)

print("\nRecords containing the unusually round amyloid maxima:")
display(
    plasma_cleaned.loc[
        plasma_cleaned["AB42_F_EQUALS_1000"]
        | plasma_cleaned["AB40_F_EQUALS_5000"],
        [
            "PHASE",
            "PTID",
            "RID",
            "VISCODE",
            "VISCODE2",
            "EXAMDATE",
            "pT217_F",
            "AB42_F",
            "AB40_F",
            "AB42_AB40_F",
            "AB42_AB40_F_RECALCULATED",
            "pT217_AB42_F",
            "pT217_AB42_F_RECALCULATED",
            "NfL_Q",
            "GFAP_Q",
            "NfL_F",
            "GFAP_F",
            "BATCH3_QC_DRIFT_VALIDATED",
            "Comment",
        ],
    ]
)

print("\nAll records flagged as possible plasma biomarker extremes:")
display(
    plasma_extreme_records.sort_values(
        ["PHASE", "RID", "EXAMDATE"]
    )
)

## 1.20. Investigate the extreme amyloid record longitudinally

The log-scale distribution review identified one particularly unusual record with Aβ42 equal to 1,000 pg/mL and Aβ40 equal to 5,000 pg/mL. These exact round values are substantially larger than the rest of the plasma measurements and may represent an assay limit, placeholder, or unusual laboratory result.

inspect all plasma records for this participant and compare the extreme measurement with the participant's other visits. also examine the closest values elsewhere in the dataset before deciding whether the amyloid measurements and their derived ratios should be retained or converted to missing values.

In [ ]:
# Identify the participant containing the unusually round amyloid values.
extreme_amyloid_rid = int(
    plasma_cleaned.loc[
        plasma_cleaned["AB42_F_EQUALS_1000"]
        | plasma_cleaned["AB40_F_EQUALS_5000"],
        "RID",
    ].iloc[0]
)

# Display every plasma record available for this participant.
participant_amyloid_history = (
    plasma_cleaned.loc[
        plasma_cleaned["RID"] == extreme_amyloid_rid,
        [
            "PHASE",
            "PTID",
            "RID",
            "VISCODE",
            "VISCODE2",
            "EXAMDATE",
            "pT217_F",
            "AB42_F",
            "AB40_F",
            "AB42_AB40_F",
            "pT217_AB42_F",
            "NfL_Q",
            "GFAP_Q",
            "NfL_F",
            "GFAP_F",
            "Comment",
        ],
    ]
    .sort_values("EXAMDATE")
)

print(
    f"All plasma records for RID {extreme_amyloid_rid}:"
)
display(participant_amyloid_history)

# Display the largest amyloid measurements in the complete dataset to show
# whether the extreme values are isolated from the rest of the distribution.
print("\nTwenty largest Aβ42 measurements:")
display(
    plasma_cleaned.loc[
        plasma_cleaned["AB42_F"].notna(),
        [
            "PHASE",
            "PTID",
            "RID",
            "VISCODE2",
            "EXAMDATE",
            "AB42_F",
            "AB40_F",
            "AB42_AB40_F",
            "pT217_F",
            "pT217_AB42_F",
            "Comment",
        ],
    ]
    .sort_values("AB42_F", ascending=False)
    .head(20)
)

print("\nTwenty largest Aβ40 measurements:")
display(
    plasma_cleaned.loc[
        plasma_cleaned["AB40_F"].notna(),
        [
            "PHASE",
            "PTID",
            "RID",
            "VISCODE2",
            "EXAMDATE",
            "AB42_F",
            "AB40_F",
            "AB42_AB40_F",
            "pT217_F",
            "pT217_AB42_F",
            "Comment",
        ],
    ]
    .sort_values("AB40_F", ascending=False)
    .head(20)
)

## 1.21. Exclude the unresolved extreme amyloid measurements

The record for RID 4136 contains exact round values of 1,000 pg/mL for Aβ42 and 5,000 pg/mL for Aβ40. These values are substantially separated from the remainder of the dataset, have no validating laboratory comment, and cannot be confirmed using another plasma visit for the same participant.

retain the record and its other plasma biomarkers, but treat the two amyloid concentrations and both ratios derived from Aβ42 as analytically unresolved. Their original values will be preserved in separate audit columns, and the cleaned analysis fields will be converted to missing values.

In [ ]:
# Identify the unresolved extreme amyloid record.
extreme_amyloid_mask = (
    (plasma_cleaned["RID"] == 4136)
    & (plasma_cleaned["AB42_F"] == 1000)
    & (plasma_cleaned["AB40_F"] == 5000)
)

# Confirm that exactly one record matches the intended QC rule.
extreme_amyloid_count = int(extreme_amyloid_mask.sum())

if extreme_amyloid_count != 1:
    raise ValueError(
        "The extreme amyloid QC rule was expected to match exactly one row, "
        f"but matched {extreme_amyloid_count} rows."
    )

# Preserve the original values for complete auditability.
amyloid_fields_to_exclude = [
    "AB42_F",
    "AB40_F",
    "AB42_AB40_F",
    "pT217_AB42_F",
]

for column in amyloid_fields_to_exclude:
    plasma_cleaned[f"{column}_ORIGINAL_QC"] = plasma_cleaned[column]

# Create an explicit row-level QC flag and reason.
plasma_cleaned["EXTREME_ROUND_AMYLOID_EXCLUDED"] = extreme_amyloid_mask

plasma_cleaned["AMYLOID_QC_REASON"] = pd.Series(
    pd.NA,
    index=plasma_cleaned.index,
    dtype="string",
)

plasma_cleaned.loc[
    extreme_amyloid_mask,
    "AMYLOID_QC_REASON",
] = (
    "Aβ42=1000 and Aβ40=5000 were isolated exact round values "
    "without laboratory validation or a repeat plasma measurement."
)

# Convert only the unresolved amyloid values and their dependent ratios
# to missing, while retaining the remaining biomarkers in the record.
plasma_cleaned.loc[
    extreme_amyloid_mask,
    amyloid_fields_to_exclude,
] = np.nan

print("Extreme amyloid QC adjustment completed.")

print("\nAffected record after cleaning:")
display(
    plasma_cleaned.loc[
        extreme_amyloid_mask,
        [
            "PHASE",
            "PTID",
            "RID",
            "VISCODE2",
            "EXAMDATE",
            "pT217_F",
            "AB42_F_ORIGINAL_QC",
            "AB40_F_ORIGINAL_QC",
            "AB42_AB40_F_ORIGINAL_QC",
            "pT217_AB42_F_ORIGINAL_QC",
            "AB42_F",
            "AB40_F",
            "AB42_AB40_F",
            "pT217_AB42_F",
            "NfL_Q",
            "GFAP_Q",
            "EXTREME_ROUND_AMYLOID_EXCLUDED",
            "AMYLOID_QC_REASON",
        ],
    ]
)

print("\nUpdated valid-value counts:")
display(
    pd.DataFrame(
        {
            "MARKER": amyloid_fields_to_exclude,
            "VALID_VALUES": [
                int(plasma_cleaned[column].notna().sum())
                for column in amyloid_fields_to_exclude
            ],
            "MISSING_VALUES": [
                int(plasma_cleaned[column].isna().sum())
                for column in amyloid_fields_to_exclude
            ],
        }
    )
)

## 1.22. Compare overlapping Fujirebio and Quanterix NfL and GFAP measurements

Some ADNI4 samples contain both Fujirebio and Quanterix measurements for NfL and GFAP. Although they measure the same underlying biomarkers, values from different assay platforms should not be assumed to be numerically interchangeable.

examine the overlapping records to assess:

- how many samples contain both platform measurements;
- the correlation between platforms;
- whether one platform is systematically higher or lower;
- whether the relationship is sufficiently stable to justify keeping the measurements as separate platform-specific features.

No values will be merged or converted at this stage.

In [ ]:
# Identify records with measurements from both platforms.
nfl_overlap_mask = (
    plasma_cleaned["NfL_Q"].notna()
    & plasma_cleaned["NfL_F"].notna()
)

gfap_overlap_mask = (
    plasma_cleaned["GFAP_Q"].notna()
    & plasma_cleaned["GFAP_F"].notna()
)

# Calculate platform-comparison statistics.
platform_comparison_summary = pd.DataFrame(
    {
        "BIOMARKER": ["NfL", "GFAP"],
        "OVERLAPPING_RECORDS": [
            int(nfl_overlap_mask.sum()),
            int(gfap_overlap_mask.sum()),
        ],
        "PEARSON_CORRELATION": [
            plasma_cleaned.loc[
                nfl_overlap_mask,
                ["NfL_Q", "NfL_F"],
            ].corr(method="pearson").iloc[0, 1],
            plasma_cleaned.loc[
                gfap_overlap_mask,
                ["GFAP_Q", "GFAP_F"],
            ].corr(method="pearson").iloc[0, 1],
        ],
        "SPEARMAN_CORRELATION": [
            plasma_cleaned.loc[
                nfl_overlap_mask,
                ["NfL_Q", "NfL_F"],
            ].corr(method="spearman").iloc[0, 1],
            plasma_cleaned.loc[
                gfap_overlap_mask,
                ["GFAP_Q", "GFAP_F"],
            ].corr(method="spearman").iloc[0, 1],
        ],
        "QUANTERIX_MEDIAN": [
            plasma_cleaned.loc[nfl_overlap_mask, "NfL_Q"].median(),
            plasma_cleaned.loc[gfap_overlap_mask, "GFAP_Q"].median(),
        ],
        "FUJIREBIO_MEDIAN": [
            plasma_cleaned.loc[nfl_overlap_mask, "NfL_F"].median(),
            plasma_cleaned.loc[gfap_overlap_mask, "GFAP_F"].median(),
        ],
    }
)

print("Cross-platform comparison summary:")
display(platform_comparison_summary)

# Calculate record-level platform ratios and absolute differences.
plasma_cleaned["NfL_F_TO_Q_RATIO"] = np.nan
plasma_cleaned["GFAP_F_TO_Q_RATIO"] = np.nan
plasma_cleaned["NfL_F_MINUS_Q"] = np.nan
plasma_cleaned["GFAP_F_MINUS_Q"] = np.nan

plasma_cleaned.loc[
    nfl_overlap_mask,
    "NfL_F_TO_Q_RATIO",
] = (
    plasma_cleaned.loc[nfl_overlap_mask, "NfL_F"]
    / plasma_cleaned.loc[nfl_overlap_mask, "NfL_Q"]
)

plasma_cleaned.loc[
    gfap_overlap_mask,
    "GFAP_F_TO_Q_RATIO",
] = (
    plasma_cleaned.loc[gfap_overlap_mask, "GFAP_F"]
    / plasma_cleaned.loc[gfap_overlap_mask, "GFAP_Q"]
)

plasma_cleaned.loc[
    nfl_overlap_mask,
    "NfL_F_MINUS_Q",
] = (
    plasma_cleaned.loc[nfl_overlap_mask, "NfL_F"]
    - plasma_cleaned.loc[nfl_overlap_mask, "NfL_Q"]
)

plasma_cleaned.loc[
    gfap_overlap_mask,
    "GFAP_F_MINUS_Q",
] = (
    plasma_cleaned.loc[gfap_overlap_mask, "GFAP_F"]
    - plasma_cleaned.loc[gfap_overlap_mask, "GFAP_Q"]
)

print("\nDistribution of Fujirebio-to-Quanterix ratios:")
display(
    plasma_cleaned.loc[
        nfl_overlap_mask | gfap_overlap_mask,
        [
            "NfL_F_TO_Q_RATIO",
            "GFAP_F_TO_Q_RATIO",
        ],
    ].describe()
)

print("\nDistribution of absolute platform differences:")
display(
    plasma_cleaned.loc[
        nfl_overlap_mask | gfap_overlap_mask,
        [
            "NfL_F_MINUS_Q",
            "GFAP_F_MINUS_Q",
        ],
    ].describe()
)

print("\nExample records with both platform measurements:")
display(
    plasma_cleaned.loc[
        nfl_overlap_mask & gfap_overlap_mask,
        [
            "PHASE",
            "PTID",
            "RID",
            "VISCODE2",
            "EXAMDATE",
            "NfL_Q",
            "NfL_F",
            "NfL_F_TO_Q_RATIO",
            "GFAP_Q",
            "GFAP_F",
            "GFAP_F_TO_Q_RATIO",
            "BATCH3_QC_DRIFT_VALIDATED",
            "Comment",
        ],
    ]
    .sort_values(["RID", "EXAMDATE"])
    .head(20)
)

## 1.23. Summary of cross-platform NfL and GFAP comparison

A total of 677 plasma records contained both Fujirebio and Quanterix measurements for NfL and GFAP, allowing direct comparison of the two assay platforms.

The measurements were strongly associated across platforms, indicating that both assays capture similar underlying biological variation. For NfL, the Pearson correlation was 0.925 and the Spearman correlation was 0.895. For GFAP, the Pearson correlation was 0.752 and the Spearman correlation was 0.888.

However, the absolute concentrations differed substantially between platforms:

- Fujirebio NfL values were typically higher than Quanterix values, with a median Fujirebio-to-Quanterix ratio of approximately 1.45.
- Fujirebio GFAP values were typically much lower than Quanterix values, with a median Fujirebio-to-Quanterix ratio of approximately 0.37.

These findings show that the assays agree reasonably well in participant ranking but are not numerically interchangeable. Therefore:

- `NfL_Q` and `NfL_F` will be retained as separate platform-specific features.
- `GFAP_Q` and `GFAP_F` will be retained as separate platform-specific features.
- Values from one platform will not be used to directly fill missing values from the other.
- Cross-platform measurements will not be averaged into a single raw biomarker value.
- Any later transformation or standardisation will be fitted separately for each platform using training data only.
- Platform availability indicators will be preserved so that the model can distinguish which assay generated each measurement.

This approach preserves the biological information shared across platforms while avoiding incorrect assumptions that their concentration scales are equivalent.

## 1.24. Create and save the cleaned longitudinal plasma dataset

construct the final cleaned plasma table for downstream cohort alignment and modelling.

The final dataset will retain:

- participant and visit identifiers;
- examination dates;
- the original sample-type fields;
- cleaned plasma biomarker measurements;
- the supplied and validated biomarker ratios;
- assay-platform availability indicators;
- the validated Batch 3 QC flag;
- the flag and explanation for the excluded extreme amyloid record;
- the original laboratory comment.

Temporary recalculation columns, numerical-difference columns, distribution outlier flags, and cross-platform comparison fields were useful for quality control but will not be included as modelling features.

This remains a longitudinal plasma dataset. Selection of a baseline or nearest-date plasma record will be performed later when it is aligned with the labelled master cohort.

In [ ]:
# Define the columns required in the final cleaned longitudinal plasma dataset.
final_plasma_columns = [
    # Participant and visit identifiers
    "PHASE",
    "PTID",
    "RID",
    "VISCODE",
    "VISCODE2",
    "EXAMDATE",

    # Sample information
    "Primary",
    "Additive",

    # Fujirebio amyloid and p-tau measurements
    "pT217_F",
    "AB42_F",
    "AB40_F",
    "AB42_AB40_F",
    "pT217_AB42_F",

    # Quanterix NfL and GFAP measurements
    "NfL_Q",
    "GFAP_Q",

    # Fujirebio NfL and GFAP measurements
    "NfL_F",
    "GFAP_F",

    # Assay-availability indicators
    "HAS_FUJIREBIO_AMYLOID_PTAU",
    "HAS_QUANTERIX_NFL_GFAP",
    "HAS_FUJIREBIO_NFL_GFAP",

    # Quality-control information
    "BATCH3_QC_DRIFT_VALIDATED",
    "EXTREME_ROUND_AMYLOID_EXCLUDED",
    "AMYLOID_QC_REASON",
    "Comment",
]

# Create the compact analysis-ready longitudinal table.
plasma_final = (
    plasma_cleaned[final_plasma_columns]
    .copy()
    .sort_values(["RID", "EXAMDATE", "PHASE", "VISCODE2"])
    .reset_index(drop=True)
)

# Define output paths.
plasma_final_path = (
    plasma_processed_dir
    / "plasma_biomarkers_cleaned_longitudinal.csv"
)

plasma_special_code_qc_path = (
    plasma_qc_dir
    / "plasma_special_code_summary.csv"
)

plasma_phase_coverage_path = (
    plasma_qc_dir
    / "plasma_phase_marker_coverage.csv"
)

plasma_distribution_qc_path = (
    plasma_qc_dir
    / "plasma_distribution_summary.csv"
)

plasma_platform_comparison_path = (
    plasma_qc_dir
    / "plasma_platform_comparison_summary.csv"
)

plasma_extreme_records_path = (
    plasma_qc_dir
    / "plasma_possible_extreme_records.csv"
)

# Save the cleaned longitudinal dataset.
plasma_final.to_csv(
    plasma_final_path,
    index=False,
)

# Save the main quality-control summaries.
special_code_summary.to_csv(
    plasma_special_code_qc_path,
    index=False,
)

phase_marker_percentages.to_csv(
    plasma_phase_coverage_path,
    index=False,
)

plasma_distribution_summary.to_csv(
    plasma_distribution_qc_path,
    index=False,
)

platform_comparison_summary.to_csv(
    plasma_platform_comparison_path,
    index=False,
)

plasma_extreme_records.to_csv(
    plasma_extreme_records_path,
    index=False,
)

# Confirm the saved dataset and its final structure.
print("Cleaned longitudinal plasma dataset saved successfully.")
print(f"\nDataset path:\n{plasma_final_path}")

print("\nFinal dataset dimensions:")
print(f"Rows: {plasma_final.shape[0]:,}")
print(f"Columns: {plasma_final.shape[1]:,}")
print(f"Participants: {plasma_final['RID'].nunique():,}")

print("\nSaved quality-control outputs:")
print(f"- {plasma_special_code_qc_path}")
print(f"- {plasma_phase_coverage_path}")
print(f"- {plasma_distribution_qc_path}")
print(f"- {plasma_platform_comparison_path}")
print(f"- {plasma_extreme_records_path}")

print("\nFinal cleaned plasma column structure:")
display(
    pd.DataFrame(
        {
            "COLUMN_NAME": plasma_final.columns,
            "DATA_TYPE": plasma_final.dtypes.astype(str).values,
            "MISSING_VALUES": [
                int(plasma_final[column].isna().sum())
                for column in plasma_final.columns
            ],
        }
    )
)

## 1.25. Load the CSF dataset and prepare the plasma data for cross-modality analysis

load the cleaned visit-level CSF biomarker dataset from its known path.

For plasma, reuse the existing cleaned working DataFrame if `plasma_cleaned` is already available in this notebook. Otherwise, locate the raw Fujirebio-Quanterix plasma CSV whose filename begins with `All_Subjects_UPENN_PLASMA_FUJIREBIO_QUANTERIX_` and load it.

At this stage, only verify:

- that both datasets are available;
- their dimensions and participant counts;
- whether all required comparison fields are present;
- whether the ratio fields contain valid positive values.

No merging or participant-level counting will be performed yet.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

# Define the cleaned CSF visit-level file.
csf_crossmodality_path = Path(
    "/content/drive/MyDrive/adni_mri/adni_non_imaging/"
    "interim/csf_core_biomarkers/"
    "csf_core_biomarkers_cleaned_visit_level.csv"
)

# Define the raw plasma source folder.
plasma_source_dir = Path(
    "/content/drive/MyDrive/adni_mri/adni_non_imaging/"
    "raw/Plasma biomarker panel"
)

# Confirm that the CSF source file exists.
if not csf_crossmodality_path.exists():
    raise FileNotFoundError(
        f"The cleaned CSF file was not found at:\n{csf_crossmodality_path}"
    )

# Load the cleaned CSF visit-level dataset.
csf_crossmodality = pd.read_csv(
    csf_crossmodality_path,
    low_memory=False,
)

# Reuse the cleaned plasma working DataFrame when it already exists.
if (
    "plasma_cleaned" in globals()
    and isinstance(plasma_cleaned, pd.DataFrame)
):
    plasma_crossmodality = plasma_cleaned.copy()
    plasma_data_source = "Existing cleaned plasma working DataFrame: plasma_cleaned"

else:
    # Locate the expected plasma source CSV.
    plasma_matches = sorted(
        plasma_source_dir.glob(
            "All_Subjects_UPENN_PLASMA_FUJIREBIO_QUANTERIX_*.csv"
        )
    )

    if len(plasma_matches) == 0:
        raise FileNotFoundError(
            "No plasma CSV beginning with "
            "'All_Subjects_UPENN_PLASMA_FUJIREBIO_QUANTERIX_' "
            f"was found in:\n{plasma_source_dir}"
        )

    if len(plasma_matches) > 1:
        print("Multiple matching plasma files were found:")
        for matched_path in plasma_matches:
            print(f"- {matched_path}")

        raise RuntimeError(
            "More than one matching plasma source file was found. "
            "Please confirm which file should be used."
        )

    plasma_source_path = plasma_matches[0]

    plasma_crossmodality = pd.read_csv(
        plasma_source_path,
        low_memory=False,
    )

    plasma_data_source = f"Raw plasma CSV: {plasma_source_path}"

# Define the required fields for this cross-modality analysis.
required_csf_fields = [
    "RID",
    "PTID",
    "VISCODE2",
    "EXAMDATE",
    "ABETA40",
    "ABETA42",
    "ABETA42_40_RATIO",
    "ABETA42_40_RATIO_AVAILABLE",
    "BATCH",
]

required_plasma_fields = [
    "RID",
    "PTID",
    "VISCODE2",
    "EXAMDATE",
    "AB42_F",
    "AB40_F",
    "AB42_AB40_F",
]

# Check for missing required columns.
missing_csf_fields = [
    column
    for column in required_csf_fields
    if column not in csf_crossmodality.columns
]

missing_plasma_fields = [
    column
    for column in required_plasma_fields
    if column not in plasma_crossmodality.columns
]

if missing_csf_fields:
    raise KeyError(
        "The following required CSF fields are missing:\n"
        f"{missing_csf_fields}"
    )

if missing_plasma_fields:
    raise KeyError(
        "The following required plasma fields are missing:\n"
        f"{missing_plasma_fields}"
    )

# Standardise identifiers and dates for later matching.
for dataframe in [csf_crossmodality, plasma_crossmodality]:
    dataframe["RID"] = pd.to_numeric(
        dataframe["RID"],
        errors="coerce",
    ).astype("Int64")

    dataframe["PTID"] = (
        dataframe["PTID"]
        .astype("string")
        .str.strip()
    )

    dataframe["VISCODE2"] = (
        dataframe["VISCODE2"]
        .astype("string")
        .str.strip()
    )

    dataframe["EXAMDATE"] = pd.to_datetime(
        dataframe["EXAMDATE"],
        errors="coerce",
    )

# Ensure ratio and component fields are numeric.
for column in [
    "ABETA40",
    "ABETA42",
    "ABETA42_40_RATIO",
]:
    csf_crossmodality[column] = pd.to_numeric(
        csf_crossmodality[column],
        errors="coerce",
    )

for column in [
    "AB42_F",
    "AB40_F",
    "AB42_AB40_F",
]:
    plasma_crossmodality[column] = pd.to_numeric(
        plasma_crossmodality[column],
        errors="coerce",
    )

# Treat documented negative plasma administrative codes as unavailable
# if the raw file had to be loaded instead of using plasma_cleaned.
for column in [
    "AB42_F",
    "AB40_F",
    "AB42_AB40_F",
]:
    plasma_crossmodality.loc[
        plasma_crossmodality[column] < 0,
        column,
    ] = np.nan

# Define valid ratio availability for the initial verification.
valid_csf_ratio_mask = (
    csf_crossmodality["ABETA42_40_RATIO"].notna()
    & (csf_crossmodality["ABETA42_40_RATIO"] > 0)
)

valid_plasma_ratio_mask = (
    plasma_crossmodality["AB42_AB40_F"].notna()
    & (plasma_crossmodality["AB42_AB40_F"] > 0)
)

# Display a compact source-verification summary.
source_verification_summary = pd.DataFrame(
    {
        "MODALITY": ["CSF", "Plasma"],
        "ROWS": [
            len(csf_crossmodality),
            len(plasma_crossmodality),
        ],
        "UNIQUE_RIDS": [
            csf_crossmodality["RID"].nunique(dropna=True),
            plasma_crossmodality["RID"].nunique(dropna=True),
        ],
        "VALID_RATIO_RECORDS": [
            int(valid_csf_ratio_mask.sum()),
            int(valid_plasma_ratio_mask.sum()),
        ],
        "MISSING_RID": [
            int(csf_crossmodality["RID"].isna().sum()),
            int(plasma_crossmodality["RID"].isna().sum()),
        ],
        "MISSING_EXAMDATE": [
            int(csf_crossmodality["EXAMDATE"].isna().sum()),
            int(plasma_crossmodality["EXAMDATE"].isna().sum()),
        ],
    }
)

print("Cross-modality source verification completed.")
print(f"\nCSF source:\n{csf_crossmodality_path}")
print(f"\nPlasma source:\n{plasma_data_source}")

print("\nDataset verification summary:")
display(source_verification_summary)

print("\nAll required CSF and plasma fields are present.")

## 1.26. Check whether plasma covers missing CSF ratios among overlapping participants

restrict the analysis to participants who appear in both the CSF and plasma datasets.

Within this overlapping group, identify participants who have no valid `ABETA42_40_RATIO` at any CSF visit but do have at least one valid plasma `AB42_AB40_F` measurement.

This directly tests whether plasma provides additional amyloid-ratio coverage for participants whose CSF records do not contain the ratio. Repeated visits will be collapsed to one participant-level result.

In [ ]:
# Restrict both datasets to participants who appear in both modalities.
csf_overlap = csf_crossmodality.loc[
    csf_crossmodality["RID"].isin(both_modality_rids)
].copy()

plasma_overlap = plasma_crossmodality.loc[
    plasma_crossmodality["RID"].isin(both_modality_rids)
].copy()

# Create row-level valid-ratio indicators.
csf_overlap["VALID_CSF_RATIO"] = (
    csf_overlap["ABETA42_40_RATIO"].notna()
    & (csf_overlap["ABETA42_40_RATIO"] > 0)
)

plasma_overlap["VALID_PLASMA_RATIO"] = (
    plasma_overlap["AB42_AB40_F"].notna()
    & (plasma_overlap["AB42_AB40_F"] > 0)
)

# Collapse the CSF data to one row per overlapping participant.
csf_overlap_participant = (
    csf_overlap
    .groupby("RID", as_index=False)
    .agg(
        PTID_CSF=("PTID", "first"),
        CSF_VISIT_COUNT=("RID", "size"),
        HAS_VALID_CSF_RATIO=("VALID_CSF_RATIO", "any"),
        CSF_FIRST_DATE=("EXAMDATE", "min"),
        CSF_LAST_DATE=("EXAMDATE", "max"),
    )
)

# Collapse the plasma data to one row per overlapping participant.
plasma_overlap_participant = (
    plasma_overlap
    .groupby("RID", as_index=False)
    .agg(
        PTID_PLASMA=("PTID", "first"),
        PLASMA_VISIT_COUNT=("RID", "size"),
        HAS_VALID_PLASMA_RATIO=("VALID_PLASMA_RATIO", "any"),
        PLASMA_FIRST_DATE=("EXAMDATE", "min"),
        PLASMA_LAST_DATE=("EXAMDATE", "max"),
    )
)

# Merge the participant-level summaries.
overlap_ratio_coverage = csf_overlap_participant.merge(
    plasma_overlap_participant,
    on="RID",
    how="inner",
    validate="one_to_one",
)

# Keep one participant identifier and verify consistency across modalities.
overlap_ratio_coverage["PTID"] = (
    overlap_ratio_coverage["PTID_CSF"]
    .combine_first(overlap_ratio_coverage["PTID_PLASMA"])
)

overlap_ratio_coverage["PTID_MATCHES"] = (
    overlap_ratio_coverage["PTID_CSF"]
    == overlap_ratio_coverage["PTID_PLASMA"]
)

# Identify overlapping participants with no valid CSF ratio
# but at least one valid plasma ratio.
plasma_fills_csf_ratio_gap_mask = (
    ~overlap_ratio_coverage["HAS_VALID_CSF_RATIO"]
    & overlap_ratio_coverage["HAS_VALID_PLASMA_RATIO"]
)

plasma_fills_csf_ratio_gap = (
    overlap_ratio_coverage.loc[
        plasma_fills_csf_ratio_gap_mask,
        [
            "RID",
            "PTID",
            "CSF_VISIT_COUNT",
            "PLASMA_VISIT_COUNT",
            "HAS_VALID_CSF_RATIO",
            "HAS_VALID_PLASMA_RATIO",
            "CSF_FIRST_DATE",
            "CSF_LAST_DATE",
            "PLASMA_FIRST_DATE",
            "PLASMA_LAST_DATE",
        ],
    ]
    .sort_values("RID")
    .reset_index(drop=True)
)

# Calculate the relevant counts.
overlap_without_csf_ratio_count = int(
    (~overlap_ratio_coverage["HAS_VALID_CSF_RATIO"]).sum()
)

gap_covered_by_plasma_count = int(
    plasma_fills_csf_ratio_gap.shape[0]
)

gap_covered_by_plasma_percentage = (
    gap_covered_by_plasma_count
    / overlap_without_csf_ratio_count
    * 100
    if overlap_without_csf_ratio_count > 0
    else np.nan
)

# Build a full, non-truncated summary table.
direct_gap_summary = pd.DataFrame(
    {
        "CHECK": [
            "Participants appearing in both CSF and plasma",
            "Overlapping participants without any valid CSF ratio",
            "Of these, participants with at least one valid plasma ratio",
            "Percentage of overlapping participants without a CSF ratio who gain plasma ratio coverage",
        ],
        "RESULT": [
            len(both_modality_rids),
            overlap_without_csf_ratio_count,
            gap_covered_by_plasma_count,
            f"{gap_covered_by_plasma_percentage:.2f}%",
        ],
    }
)

# Print the full summary table without truncation.
print("Does plasma cover missing CSF ratios among overlapping participants?\n")
print(
    direct_gap_summary.to_string(
        index=False,
        justify="left",
    )
)

print(
    "\nInterpretation:\n"
    f"Among the {len(both_modality_rids):,} participants who appear in both datasets, "
    f"{overlap_without_csf_ratio_count:,} have no valid CSF Aβ42/Aβ40 ratio at any visit. "
    f"All {gap_covered_by_plasma_count:,} of these participants have at least one valid plasma ratio, "
    f"so plasma provides ratio coverage for {gap_covered_by_plasma_percentage:.2f}% of this group."
)

# Display the participant-level table fully.
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", None)

print(
    "\nParticipants with no valid CSF ratio but at least one valid plasma ratio:\n"
)

display(plasma_fills_csf_ratio_gap)

# Confirm PTID agreement across modalities.
if not overlap_ratio_coverage["PTID_MATCHES"].all():
    print("\nWarning: Some overlapping RIDs have different PTIDs between modalities.")
    display(
        overlap_ratio_coverage.loc[
            ~overlap_ratio_coverage["PTID_MATCHES"],
            ["RID", "PTID_CSF", "PTID_PLASMA"],
        ]
    )
else:
    print("\nPTID values agree for all overlapping participants.")

## 1.27. Check whether plasma ratio dates align with CSF Aβ42 dates for the CSF-ratio gap group

focus on the 253 participants who appear in both modalities, have no valid CSF Aβ42/Aβ40 ratio at any visit, but have at least one valid plasma Aβ42/Aβ40 ratio.

For each CSF record containing a valid `ABETA42` measurement, select the valid plasma `AB42_AB40_F` record from the same participant with the nearest examination date. This avoids an uncontrolled many-to-many merge when participants have repeated visits.

summarise how many nearest matches occur:

- on the exact same date;
- within 30 days;
- within 90 days;
- within 365 days;
- more than 365 days apart.

This will determine whether the additional plasma-ratio coverage represents measurements collected at approximately the same time as the available CSF Aβ42 measurement.

In [ ]:
# Extract the RIDs for participants who have no valid CSF ratio
# but do have at least one valid plasma Aβ42/Aβ40 ratio.
csf_gap_with_plasma_ratio_rids = set(
    plasma_fills_csf_ratio_gap["RID"]
    .dropna()
    .astype(int)
)

# Retain valid CSF Aβ42 records for the 253 participants.
csf_abeta42_gap_records = (
    csf_crossmodality.loc[
        csf_crossmodality["RID"].isin(csf_gap_with_plasma_ratio_rids)
        & csf_crossmodality["ABETA42"].notna()
        & (csf_crossmodality["ABETA42"] > 0),
        [
            "RID",
            "PTID",
            "VISCODE2",
            "EXAMDATE",
            "ABETA42",
            "ABETA40",
            "ABETA42_40_RATIO",
            "BATCH",
        ],
    ]
    .copy()
    .rename(
        columns={
            "PTID": "PTID_CSF",
            "VISCODE2": "VISCODE2_CSF",
            "EXAMDATE": "EXAMDATE_CSF",
        }
    )
    .reset_index(drop=True)
)

# Retain valid plasma Aβ42/Aβ40 ratio records for the same participants.
plasma_ratio_gap_records = (
    plasma_crossmodality.loc[
        plasma_crossmodality["RID"].isin(csf_gap_with_plasma_ratio_rids)
        & plasma_crossmodality["AB42_AB40_F"].notna()
        & (plasma_crossmodality["AB42_AB40_F"] > 0),
        [
            "RID",
            "PTID",
            "VISCODE2",
            "EXAMDATE",
            "AB42_F",
            "AB40_F",
            "AB42_AB40_F",
        ],
    ]
    .copy()
    .rename(
        columns={
            "PTID": "PTID_PLASMA",
            "VISCODE2": "VISCODE2_PLASMA",
            "EXAMDATE": "EXAMDATE_PLASMA",
        }
    )
)

# Give every CSF Aβ42 record a stable identifier.
csf_abeta42_gap_records["CSF_ABETA42_RECORD_ID"] = (
    csf_abeta42_gap_records.index
)

# Generate candidate pairs only within the same participant.
candidate_gap_date_pairs = csf_abeta42_gap_records.merge(
    plasma_ratio_gap_records,
    on="RID",
    how="inner",
    validate="many_to_many",
)

# Calculate the absolute number of days between the CSF Aβ42
# collection and plasma ratio collection.
candidate_gap_date_pairs["DATE_DIFFERENCE_DAYS"] = (
    candidate_gap_date_pairs["EXAMDATE_CSF"]
    - candidate_gap_date_pairs["EXAMDATE_PLASMA"]
).abs().dt.days

candidate_gap_date_pairs["SAME_VISCODE2"] = (
    candidate_gap_date_pairs["VISCODE2_CSF"]
    == candidate_gap_date_pairs["VISCODE2_PLASMA"]
)

# For each CSF Aβ42 record, select the nearest plasma ratio record.
# Ties are resolved by preferring the same VISCODE2 and then
# the earlier plasma examination date.
nearest_plasma_ratio_to_csf_abeta42 = (
    candidate_gap_date_pairs
    .sort_values(
        [
            "CSF_ABETA42_RECORD_ID",
            "DATE_DIFFERENCE_DAYS",
            "SAME_VISCODE2",
            "EXAMDATE_PLASMA",
        ],
        ascending=[True, True, False, True],
    )
    .drop_duplicates(
        subset="CSF_ABETA42_RECORD_ID",
        keep="first",
    )
    .reset_index(drop=True)
)

# Create non-overlapping date-alignment categories.
nearest_plasma_ratio_to_csf_abeta42["DATE_ALIGNMENT_CATEGORY"] = pd.cut(
    nearest_plasma_ratio_to_csf_abeta42["DATE_DIFFERENCE_DAYS"],
    bins=[-1, 0, 30, 90, 365, np.inf],
    labels=[
        "Exact same date",
        "1–30 days apart",
        "31–90 days apart",
        "91–365 days apart",
        "More than 365 days apart",
    ],
)

# Count unique participants as well as matched CSF records.
date_alignment_summary = (
    nearest_plasma_ratio_to_csf_abeta42
    .groupby(
        "DATE_ALIGNMENT_CATEGORY",
        observed=False,
    )
    .agg(
        MATCHED_CSF_RECORDS=("CSF_ABETA42_RECORD_ID", "size"),
        UNIQUE_PARTICIPANTS=("RID", "nunique"),
    )
    .reset_index()
)

# Add cumulative thresholds requested for interpretation.
cumulative_date_summary = pd.DataFrame(
    {
        "DATE_ALIGNMENT_CHECK": [
            "Exact same examination date",
            "Within 30 days, including exact-date matches",
            "Within 90 days, including exact-date matches",
            "Within 365 days, including exact-date matches",
            "More than 365 days apart",
        ],
        "MATCHED_CSF_RECORDS": [
            int(
                (
                    nearest_plasma_ratio_to_csf_abeta42[
                        "DATE_DIFFERENCE_DAYS"
                    ] == 0
                ).sum()
            ),
            int(
                (
                    nearest_plasma_ratio_to_csf_abeta42[
                        "DATE_DIFFERENCE_DAYS"
                    ] <= 30
                ).sum()
            ),
            int(
                (
                    nearest_plasma_ratio_to_csf_abeta42[
                        "DATE_DIFFERENCE_DAYS"
                    ] <= 90
                ).sum()
            ),
            int(
                (
                    nearest_plasma_ratio_to_csf_abeta42[
                        "DATE_DIFFERENCE_DAYS"
                    ] <= 365
                ).sum()
            ),
            int(
                (
                    nearest_plasma_ratio_to_csf_abeta42[
                        "DATE_DIFFERENCE_DAYS"
                    ] > 365
                ).sum()
            ),
        ],
        "UNIQUE_PARTICIPANTS": [
            nearest_plasma_ratio_to_csf_abeta42.loc[
                nearest_plasma_ratio_to_csf_abeta42[
                    "DATE_DIFFERENCE_DAYS"
                ] == 0,
                "RID",
            ].nunique(),
            nearest_plasma_ratio_to_csf_abeta42.loc[
                nearest_plasma_ratio_to_csf_abeta42[
                    "DATE_DIFFERENCE_DAYS"
                ] <= 30,
                "RID",
            ].nunique(),
            nearest_plasma_ratio_to_csf_abeta42.loc[
                nearest_plasma_ratio_to_csf_abeta42[
                    "DATE_DIFFERENCE_DAYS"
                ] <= 90,
                "RID",
            ].nunique(),
            nearest_plasma_ratio_to_csf_abeta42.loc[
                nearest_plasma_ratio_to_csf_abeta42[
                    "DATE_DIFFERENCE_DAYS"
                ] <= 365,
                "RID",
            ].nunique(),
            nearest_plasma_ratio_to_csf_abeta42.loc[
                nearest_plasma_ratio_to_csf_abeta42[
                    "DATE_DIFFERENCE_DAYS"
                ] > 365,
                "RID",
            ].nunique(),
        ],
    }
)

print(
    "Plasma Aβ42/Aβ40 ratio date alignment with available "
    "CSF Aβ42 measurements:\n"
)
print(cumulative_date_summary.to_string(index=False))

print("\nNon-overlapping date-separation categories:\n")
print(date_alignment_summary.to_string(index=False))

print("\nDate-difference distribution:\n")
print(
    nearest_plasma_ratio_to_csf_abeta42[
        "DATE_DIFFERENCE_DAYS"
    ]
    .describe()
    .to_string()
)

print("\nNearest-date matched records:\n")
display(
    nearest_plasma_ratio_to_csf_abeta42[
        [
            "RID",
            "PTID_CSF",
            "VISCODE2_CSF",
            "EXAMDATE_CSF",
            "ABETA42",
            "ABETA40",
            "ABETA42_40_RATIO",
            "BATCH",
            "PTID_PLASMA",
            "VISCODE2_PLASMA",
            "EXAMDATE_PLASMA",
            "AB42_F",
            "AB40_F",
            "AB42_AB40_F",
            "DATE_DIFFERENCE_DAYS",
            "SAME_VISCODE2",
            "DATE_ALIGNMENT_CATEGORY",
        ]
    ]
    .sort_values(
        ["DATE_DIFFERENCE_DAYS", "RID"]
    )
    .reset_index(drop=True)
)

## 1.28. Compare CSF and plasma Aβ42/Aβ40 ratios in participants with both measurements

identify participants who have at least one valid Aβ42/Aβ40 ratio in both the CSF and plasma datasets.

Because participants may have multiple longitudinal records, not perform a many-to-many merge. For each valid CSF ratio record, select the plasma ratio record from the same participant with the smallest absolute examination-date difference.

The resulting matched table will show whether the CSF and plasma ratios are numerically identical. This is a comparison only: the two ratios come from different biological specimens and must not be treated as interchangeable.

In [ ]:
# Retain only valid ratio records from participants appearing in both modalities.
csf_ratio_records = (
    csf_crossmodality.loc[
        csf_crossmodality["RID"].isin(both_modality_rids)
        & csf_crossmodality["ABETA42_40_RATIO"].notna()
        & (csf_crossmodality["ABETA42_40_RATIO"] > 0),
        [
            "RID",
            "PTID",
            "VISCODE2",
            "EXAMDATE",
            "ABETA42",
            "ABETA40",
            "ABETA42_40_RATIO",
            "BATCH",
        ],
    ]
    .copy()
    .rename(
        columns={
            "PTID": "PTID_CSF",
            "VISCODE2": "VISCODE2_CSF",
            "EXAMDATE": "EXAMDATE_CSF",
        }
    )
)

plasma_ratio_records = (
    plasma_crossmodality.loc[
        plasma_crossmodality["RID"].isin(both_modality_rids)
        & plasma_crossmodality["AB42_AB40_F"].notna()
        & (plasma_crossmodality["AB42_AB40_F"] > 0),
        [
            "RID",
            "PTID",
            "VISCODE2",
            "EXAMDATE",
            "AB42_F",
            "AB40_F",
            "AB42_AB40_F",
        ],
    ]
    .copy()
    .rename(
        columns={
            "PTID": "PTID_PLASMA",
            "VISCODE2": "VISCODE2_PLASMA",
            "EXAMDATE": "EXAMDATE_PLASMA",
        }
    )
)

# Identify participants with a valid ratio in both modalities.
rids_with_both_valid_ratios = (
    set(csf_ratio_records["RID"].dropna().astype(int))
    & set(plasma_ratio_records["RID"].dropna().astype(int))
)

csf_ratio_records = csf_ratio_records.loc[
    csf_ratio_records["RID"].isin(rids_with_both_valid_ratios)
].copy()

plasma_ratio_records = plasma_ratio_records.loc[
    plasma_ratio_records["RID"].isin(rids_with_both_valid_ratios)
].copy()

# Create a stable identifier for each CSF record.
csf_ratio_records = csf_ratio_records.reset_index(drop=True)
csf_ratio_records["CSF_RECORD_ID"] = csf_ratio_records.index

# Generate candidate pairs only within the same participant.
candidate_ratio_pairs = csf_ratio_records.merge(
    plasma_ratio_records,
    on="RID",
    how="inner",
    validate="many_to_many",
)

# Calculate absolute examination-date separation.
candidate_ratio_pairs["DATE_DIFFERENCE_DAYS"] = (
    candidate_ratio_pairs["EXAMDATE_CSF"]
    - candidate_ratio_pairs["EXAMDATE_PLASMA"]
).abs().dt.days

# Prefer the nearest date. When two plasma records are equally close,
# prefer the same VISCODE2, then the earlier plasma date.
candidate_ratio_pairs["SAME_VISCODE2"] = (
    candidate_ratio_pairs["VISCODE2_CSF"]
    == candidate_ratio_pairs["VISCODE2_PLASMA"]
)

nearest_ratio_matches = (
    candidate_ratio_pairs
    .sort_values(
        [
            "CSF_RECORD_ID",
            "DATE_DIFFERENCE_DAYS",
            "SAME_VISCODE2",
            "EXAMDATE_PLASMA",
        ],
        ascending=[True, True, False, True],
    )
    .drop_duplicates(
        subset="CSF_RECORD_ID",
        keep="first",
    )
    .reset_index(drop=True)
)

# Compare the CSF and plasma ratio values.
nearest_ratio_matches["EXACTLY_EQUAL"] = (
    nearest_ratio_matches["ABETA42_40_RATIO"]
    == nearest_ratio_matches["AB42_AB40_F"]
)

nearest_ratio_matches["EQUAL_AT_6DP"] = (
    nearest_ratio_matches["ABETA42_40_RATIO"].round(6)
    == nearest_ratio_matches["AB42_AB40_F"].round(6)
)

nearest_ratio_matches["EQUAL_AT_4DP"] = (
    nearest_ratio_matches["ABETA42_40_RATIO"].round(4)
    == nearest_ratio_matches["AB42_AB40_F"].round(4)
)

nearest_ratio_matches["EQUAL_AT_3DP"] = (
    nearest_ratio_matches["ABETA42_40_RATIO"].round(3)
    == nearest_ratio_matches["AB42_AB40_F"].round(3)
)

nearest_ratio_matches["ABSOLUTE_RATIO_DIFFERENCE"] = (
    nearest_ratio_matches["ABETA42_40_RATIO"]
    - nearest_ratio_matches["AB42_AB40_F"]
).abs()

nearest_ratio_matches["RELATIVE_RATIO_DIFFERENCE"] = (
    nearest_ratio_matches["ABSOLUTE_RATIO_DIFFERENCE"]
    / nearest_ratio_matches["ABETA42_40_RATIO"].abs()
)

# Build a full, non-truncated summary.
ratio_equality_summary = pd.DataFrame(
    {
        "CHECK": [
            "Participants with at least one valid ratio in both modalities",
            "Valid CSF ratio records assigned a nearest plasma record",
            "Ratio pairs measured on the exact same date",
            "CSF and plasma ratios exactly equal",
            "Equal after rounding to 6 decimal places",
            "Equal after rounding to 4 decimal places",
            "Equal after rounding to 3 decimal places",
        ],
        "RESULT": [
            len(rids_with_both_valid_ratios),
            len(nearest_ratio_matches),
            int((nearest_ratio_matches["DATE_DIFFERENCE_DAYS"] == 0).sum()),
            int(nearest_ratio_matches["EXACTLY_EQUAL"].sum()),
            int(nearest_ratio_matches["EQUAL_AT_6DP"].sum()),
            int(nearest_ratio_matches["EQUAL_AT_4DP"].sum()),
            int(nearest_ratio_matches["EQUAL_AT_3DP"].sum()),
        ],
    }
)

print("CSF versus plasma Aβ42/Aβ40 ratio comparison:\n")
print(ratio_equality_summary.to_string(index=False))

print("\nDifference summary:")
print(
    nearest_ratio_matches[
        [
            "DATE_DIFFERENCE_DAYS",
            "ABSOLUTE_RATIO_DIFFERENCE",
            "RELATIVE_RATIO_DIFFERENCE",
        ]
    ]
    .describe()
    .to_string()
)

print("\nNearest-date matched examples:")
display(
    nearest_ratio_matches[
        [
            "RID",
            "PTID_CSF",
            "VISCODE2_CSF",
            "EXAMDATE_CSF",
            "ABETA42",
            "ABETA40",
            "ABETA42_40_RATIO",
            "BATCH",
            "PTID_PLASMA",
            "VISCODE2_PLASMA",
            "EXAMDATE_PLASMA",
            "AB42_F",
            "AB40_F",
            "AB42_AB40_F",
            "DATE_DIFFERENCE_DAYS",
            "SAME_VISCODE2",
            "EXACTLY_EQUAL",
            "EQUAL_AT_4DP",
            "ABSOLUTE_RATIO_DIFFERENCE",
            "RELATIVE_RATIO_DIFFERENCE",
        ]
    ]
    .sort_values(
        ["DATE_DIFFERENCE_DAYS", "RID"]
    )
    .head(30)
)

## 1.29. Measure the association between CSF and plasma Aβ42/Aβ40 ratios

The nearest-date comparison showed that CSF and plasma Aβ42/Aβ40 ratios are not numerically identical, including when both samples were collected on the same examination date.

calculate Pearson and Spearman correlations for:

- all nearest-date matched records;
- exact same-date records;
- records within 30 days;
- records within 90 days;
- records within 365 days.

The same CSF participant may contribute more than one longitudinal record to these visit-level summaries. Therefore, also create a participant-level exact-date comparison using one exact-date pair per RID, selected as the earliest available exact-date match.

These analyses assess association only. They do not justify replacing, averaging, or merging the two ratios into one feature.

In [ ]:
# Define a helper function for calculating correlation statistics safely.
def calculate_ratio_correlations(dataframe, group_name):
    valid_pairs = dataframe.loc[
        dataframe["ABETA42_40_RATIO"].notna()
        & dataframe["AB42_AB40_F"].notna(),
        [
            "RID",
            "ABETA42_40_RATIO",
            "AB42_AB40_F",
        ],
    ].copy()

    if len(valid_pairs) < 2:
        pearson_correlation = np.nan
        spearman_correlation = np.nan
    else:
        pearson_correlation = valid_pairs[
            ["ABETA42_40_RATIO", "AB42_AB40_F"]
        ].corr(method="pearson").iloc[0, 1]

        spearman_correlation = valid_pairs[
            ["ABETA42_40_RATIO", "AB42_AB40_F"]
        ].corr(method="spearman").iloc[0, 1]

    return {
        "MATCHING_GROUP": group_name,
        "MATCHED_RECORDS": len(valid_pairs),
        "UNIQUE_PARTICIPANTS": valid_pairs["RID"].nunique(),
        "PEARSON_CORRELATION": pearson_correlation,
        "SPEARMAN_CORRELATION": spearman_correlation,
        "CSF_RATIO_MEDIAN": valid_pairs["ABETA42_40_RATIO"].median(),
        "PLASMA_RATIO_MEDIAN": valid_pairs["AB42_AB40_F"].median(),
    }


# Create visit-level date-alignment subsets.
exact_date_matches = nearest_ratio_matches.loc[
    nearest_ratio_matches["DATE_DIFFERENCE_DAYS"] == 0
].copy()

within_30_day_matches = nearest_ratio_matches.loc[
    nearest_ratio_matches["DATE_DIFFERENCE_DAYS"] <= 30
].copy()

within_90_day_matches = nearest_ratio_matches.loc[
    nearest_ratio_matches["DATE_DIFFERENCE_DAYS"] <= 90
].copy()

within_365_day_matches = nearest_ratio_matches.loc[
    nearest_ratio_matches["DATE_DIFFERENCE_DAYS"] <= 365
].copy()

# Calculate visit-level correlations at each date-alignment threshold.
visit_level_correlation_summary = pd.DataFrame(
    [
        calculate_ratio_correlations(
            nearest_ratio_matches,
            "All nearest-date matches",
        ),
        calculate_ratio_correlations(
            exact_date_matches,
            "Exact same date",
        ),
        calculate_ratio_correlations(
            within_30_day_matches,
            "Within 30 days",
        ),
        calculate_ratio_correlations(
            within_90_day_matches,
            "Within 90 days",
        ),
        calculate_ratio_correlations(
            within_365_day_matches,
            "Within 365 days",
        ),
    ]
)

# Create one exact-date pair per participant for a participant-level check.
# When a participant has multiple exact-date pairs, retain the earliest one.
participant_exact_date_matches = (
    exact_date_matches
    .sort_values(
        [
            "RID",
            "EXAMDATE_CSF",
            "EXAMDATE_PLASMA",
        ]
    )
    .drop_duplicates(
        subset="RID",
        keep="first",
    )
    .reset_index(drop=True)
)

participant_level_exact_summary = pd.DataFrame(
    [
        calculate_ratio_correlations(
            participant_exact_date_matches,
            "One exact-date pair per participant",
        )
    ]
)

print("Visit-level CSF–plasma ratio correlation by date alignment:\n")
print(
    visit_level_correlation_summary.to_string(
        index=False,
        float_format=lambda value: f"{value:.6f}",
    )
)

print("\nParticipant-level exact-date comparison:\n")
print(
    participant_level_exact_summary.to_string(
        index=False,
        float_format=lambda value: f"{value:.6f}",
    )
)

print("\nExact-date matched ratio examples:\n")
display(
    exact_date_matches[
        [
            "RID",
            "PTID_CSF",
            "VISCODE2_CSF",
            "EXAMDATE_CSF",
            "ABETA42",
            "ABETA40",
            "ABETA42_40_RATIO",
            "BATCH",
            "AB42_F",
            "AB40_F",
            "AB42_AB40_F",
            "ABSOLUTE_RATIO_DIFFERENCE",
            "RELATIVE_RATIO_DIFFERENCE",
        ]
    ]
    .sort_values(
        "ABSOLUTE_RATIO_DIFFERENCE",
        ascending=False,
    )
    .reset_index(drop=True)
)

## 1.30. Interpretation of the CSF-plasma Aβ42/Aβ40 comparison

The comparison supports the biological plausibility of the two measurements while confirming that they must not be treated as the same feature.

Among participants with an exact-date CSF and plasma measurement, the Aβ42/Aβ40 ratios showed a moderately strong positive association. Using one exact-date pair per participant, the Pearson correlation was approximately 0.73 and the Spearman correlation was approximately 0.74. This indicates that participants with relatively higher CSF ratios also tended to have relatively higher plasma ratios.

However, the numerical values were not equivalent. The median CSF ratio was approximately 0.060, whereas the median plasma ratio was approximately 0.086. Across all nearest-date matched records, none of the CSF and plasma ratios were exactly equal. Even measurements collected on the same date commonly showed clear absolute and relative differences.

This pattern is medically plausible because CSF and plasma represent different biological compartments. CSF is more directly connected to processes occurring in the central nervous system, whereas plasma measurements may additionally be affected by peripheral amyloid production, systemic clearance, dilution, kidney and liver function, and other physiological influences. The measurements were also produced using different laboratory platforms and concentration scales.

The reduction in Pearson correlation when wider date windows were included should be interpreted cautiously. Those comparisons included measurements collected months or years apart, repeated observations from the same participants, and CSF results from different assay batches. Therefore, the exact-date participant-level comparison provides the most clinically meaningful exploratory result.

Overall, the findings suggest that CSF and plasma Aβ42/Aβ40 are related indicators of amyloid biology, but they are not interchangeable measurements. Plasma provides complementary participant coverage where the CSF ratio is unavailable, but it must not be used to replace or directly impute the missing CSF ratio.

For modelling:

- `ABETA42_40_RATIO` should remain a CSF-specific feature;
- `AB42_AB40_F` should remain a plasma-specific feature;
- the two ratios should not be averaged;
- missing CSF ratios should not be filled using plasma ratios;
- normalization should be performed separately for each modality using parameters fitted only on the training data.

This analysis provides exploratory cross-modality validation. It does not establish clinical equivalence or define diagnostic thresholds for either measurement.

## 1.31. Save the cleaned plasma dataset and final CSF-plasma QC outputs

save the completed plasma preprocessing outputs.

The cleaned plasma visit-level dataset will be stored in the processed plasma folder. The cross-modality coverage table, nearest-date CSF-plasma comparison, and concise QC summary will be stored in the plasma QC folder.

The participant-level coverage file will retain one row per participant appearing in both datasets and will record whether each participant has a valid CSF or plasma Aβ42/Aβ40 ratio.

The nearest-date comparison file will retain the matched CSF and plasma ratio records used to assess date alignment, numerical equality, and cross-modality association.

In [ ]:
# Define the final output folders.
processed_plasma_dir = Path(
    "/content/drive/MyDrive/adni_mri/adni_non_imaging/"
    "processed/plasma_biomarker_panel"
)

plasma_qc_dir = Path(
    "/content/drive/MyDrive/adni_mri/adni_non_imaging/"
    "qc/plasma_biomarker_panel"
)

processed_plasma_dir.mkdir(parents=True, exist_ok=True)
plasma_qc_dir.mkdir(parents=True, exist_ok=True)

# Define final output file paths.
cleaned_plasma_path = (
    processed_plasma_dir
    / "plasma_biomarker_panel_cleaned_visit_level.csv"
)

participant_coverage_path = (
    plasma_qc_dir
    / "csf_plasma_participant_ratio_coverage.csv"
)

nearest_date_comparison_path = (
    plasma_qc_dir
    / "csf_plasma_abeta42_40_nearest_date_comparison.csv"
)

qc_summary_path = (
    plasma_qc_dir
    / "plasma_biomarker_panel_qc_summary.csv"
)

# Prepare the cleaned plasma dataset for export.
plasma_cleaned_export = plasma_crossmodality.copy()

plasma_cleaned_export = plasma_cleaned_export.sort_values(
    ["RID", "EXAMDATE", "VISCODE2"],
    na_position="last",
).reset_index(drop=True)

# Prepare the participant-level CSF-plasma coverage table.
participant_coverage_export = (
    overlap_ratio_coverage[
        [
            "RID",
            "PTID",
            "CSF_VISIT_COUNT",
            "PLASMA_VISIT_COUNT",
            "HAS_VALID_CSF_RATIO",
            "HAS_VALID_PLASMA_RATIO",
            "CSF_FIRST_DATE",
            "CSF_LAST_DATE",
            "PLASMA_FIRST_DATE",
            "PLASMA_LAST_DATE",
            "PTID_MATCHES",
        ]
    ]
    .copy()
    .sort_values("RID")
    .reset_index(drop=True)
)

participant_coverage_export["PLASMA_COVERS_MISSING_CSF_RATIO"] = (
    ~participant_coverage_export["HAS_VALID_CSF_RATIO"]
    & participant_coverage_export["HAS_VALID_PLASMA_RATIO"]
)

# Prepare the nearest-date matched ratio comparison.
nearest_date_comparison_export = (
    nearest_ratio_matches.copy()
    .sort_values(
        ["RID", "EXAMDATE_CSF", "EXAMDATE_PLASMA"]
    )
    .reset_index(drop=True)
)

# Build a concise QC summary containing the final key results.
qc_summary_export = pd.DataFrame(
    {
        "QC_METRIC": [
            "Raw or working plasma records",
            "Unique plasma participants",
            "Participants appearing in both CSF and plasma",
            "Overlapping participants without a valid CSF ratio",
            "Overlapping participants without a CSF ratio but with a valid plasma ratio",
            "Percentage of the overlapping CSF-ratio gap group covered by plasma",
            "Participants with a valid ratio in both modalities",
            "Nearest-date matched ratio records",
            "Exact-date matched ratio records",
            "Exactly equal CSF and plasma ratios",
            "Equal after rounding to 4 decimal places",
            "Exact-date participant-level Pearson correlation",
            "Exact-date participant-level Spearman correlation",
            "Median exact-date CSF Aβ42/Aβ40 ratio",
            "Median exact-date plasma Aβ42/Aβ40 ratio",
            "Extreme rounded plasma amyloid records excluded",
        ],
        "VALUE": [
            len(plasma_cleaned_export),
            plasma_cleaned_export["RID"].nunique(),
            len(both_modality_rids),
            overlap_without_csf_ratio_count,
            gap_covered_by_plasma_count,
            round(gap_covered_by_plasma_percentage, 2),
            len(rids_with_both_valid_ratios),
            len(nearest_ratio_matches),
            int(
                (
                    nearest_ratio_matches[
                        "DATE_DIFFERENCE_DAYS"
                    ] == 0
                ).sum()
            ),
            int(nearest_ratio_matches["EXACTLY_EQUAL"].sum()),
            int(nearest_ratio_matches["EQUAL_AT_4DP"].sum()),
            round(
                float(
                    participant_level_exact_summary.loc[
                        0,
                        "PEARSON_CORRELATION",
                    ]
                ),
                6,
            ),
            round(
                float(
                    participant_level_exact_summary.loc[
                        0,
                        "SPEARMAN_CORRELATION",
                    ]
                ),
                6,
            ),
            round(
                float(
                    participant_level_exact_summary.loc[
                        0,
                        "CSF_RATIO_MEDIAN",
                    ]
                ),
                6,
            ),
            round(
                float(
                    participant_level_exact_summary.loc[
                        0,
                        "PLASMA_RATIO_MEDIAN",
                    ]
                ),
                6,
            ),
            int(
                plasma_cleaned_export[
                    "EXTREME_ROUND_AMYLOID_EXCLUDED"
                ].fillna(False).sum()
            )
            if "EXTREME_ROUND_AMYLOID_EXCLUDED"
            in plasma_cleaned_export.columns
            else np.nan,
        ],
    }
)

qc_summary_export["INTERPRETATION"] = [
    "Final cleaned longitudinal plasma records.",
    "Unique RIDs represented in the cleaned plasma table.",
    "Participants represented in both biomarker modalities.",
    "Overlapping participants lacking a valid CSF Aβ42/Aβ40 ratio at every CSF visit.",
    "These participants gain participant-level Aβ42/Aβ40 coverage from plasma.",
    "Percentage of the overlapping CSF-ratio gap group with a valid plasma ratio.",
    "Participants eligible for direct CSF–plasma ratio comparison.",
    "Each valid CSF ratio record was assigned the nearest valid plasma ratio record from the same participant.",
    "Matched records collected on the same examination date.",
    "The two modality-specific ratios were never exactly identical.",
    "Rare rounded agreement does not establish measurement equivalence.",
    "Linear association using one exact-date pair per participant.",
    "Rank-based association using one exact-date pair per participant.",
    "Central value of exact-date CSF ratios.",
    "Central value of exact-date plasma ratios.",
    "Suspicious extreme rounded amyloid records removed from plasma analysis.",
]

# Save all final outputs.
plasma_cleaned_export.to_csv(
    cleaned_plasma_path,
    index=False,
)

participant_coverage_export.to_csv(
    participant_coverage_path,
    index=False,
)

nearest_date_comparison_export.to_csv(
    nearest_date_comparison_path,
    index=False,
)

qc_summary_export.to_csv(
    qc_summary_path,
    index=False,
)

# Confirm that every file was created successfully.
saved_files = pd.DataFrame(
    {
        "OUTPUT": [
            "Cleaned plasma visit-level dataset",
            "Participant-level CSF–plasma ratio coverage",
            "Nearest-date CSF–plasma ratio comparison",
            "Final plasma QC summary",
        ],
        "PATH": [
            str(cleaned_plasma_path),
            str(participant_coverage_path),
            str(nearest_date_comparison_path),
            str(qc_summary_path),
        ],
        "FILE_EXISTS": [
            cleaned_plasma_path.exists(),
            participant_coverage_path.exists(),
            nearest_date_comparison_path.exists(),
            qc_summary_path.exists(),
        ],
        "ROWS_SAVED": [
            len(plasma_cleaned_export),
            len(participant_coverage_export),
            len(nearest_date_comparison_export),
            len(qc_summary_export),
        ],
    }
)

print("Final plasma preprocessing outputs:\n")
print(saved_files.to_string(index=False))

assert saved_files["FILE_EXISTS"].all(), (
    "At least one plasma output file was not saved successfully."
)

print("\nFinal plasma QC summary:\n")
print(qc_summary_export.to_string(index=False))

print("\nPlasma preprocessing and CSF–plasma QC are complete.")